In [ ]:
import os
from glob import glob
import inspect

import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats, integrate, optimize
from scipy.stats import fisher_exact

from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import seaborn as sns

import cartopy.crs as ccrs
from pyproj import Proj, Transformer

from tqdm import tqdm
from iminuit import Minuit
from prettytable import PrettyTable
from dask.diagnostics import ProgressBar

def chi2_minimization(func, x, y, y_err=None, initial_params=None):
    if initial_params is None or not isinstance(initial_params, (list, tuple)):
        raise ValueError("Initial guesses for parameters must be provided as a list or tuple.")

    param_names = inspect.signature(func).parameters
    param_names = list(param_names.keys())[1:]

    if len(param_names) != len(initial_params):
        raise ValueError(f"Number of initial parameters ({len(initial_params)}) "
                         f"does not match the function parameters ({len(param_names)}).")

    if y_err is None:
        def chi2(*params):
            residuals = y - func(x, *params)
            return np.sum(residuals**2)
    else:
        def chi2(*params):
            residuals = (y - func(x, *params)) / y_err
            return np.sum(residuals**2)

    initial_param_dict = {name: val for name, val in zip(param_names, initial_params)}

    minuit = Minuit(chi2, name=param_names, **initial_param_dict)
    minuit.errordef = Minuit.LEAST_SQUARES
    minuit.migrad()

    return minuit



def calc_cov_matrix(data, ddof = 1):
    """assuming that each column represents a separate variable"""
    rows, cols = data.shape
    cov_matrix = np.empty([cols, cols])

    for i in range(cols):
        for j in range(i, cols):
            if ddof == 0:
                cov_matrix[i,j] = np.mean(data[:,i] * data[:,j]) - data[:,i].mean() * data[:,j].mean()
                cov_matrix[j,i] = cov_matrix[i,j]
            elif ddof == 1:
                cov_matrix[i,j] = 1/(rows - 1) * np.sum((data[:,i] - data[:,i].mean())*(data[:,j] - data[:,j].mean()))
                cov_matrix[j,i] = cov_matrix[i,j]
            else:
                print("The degrees of freedom must be 0 or 1")
                return None

    return cov_matrix

def FIT_DATA(x, y, x_err, y_err, model, initial_params, fit_label, xlabel, ylabel, title, text_x=1.05, text_y=0.9, fontsize=15):
    # ------------ DEFINE DATA ------------
    xs = x
    ys = y
    sy = y_err
    sx = x_err

    x_linspace = np.linspace(xs.min(), xs.max(),  len(xs))

    if len(y_err) == 1:
        sy = np.ones_like(x_linspace) * sy
    # ------------ FIT ------------
    minuit = chi2_minimization(model, xs, ys, y_err=sy, initial_params=initial_params)

    if not minuit.fmin.is_valid:
        print('WARNING: The ChiSquare fit DID NOT converge!!!')

    npara = len(initial_params)
    ndof_fit = len(xs) - npara

    chi2_fit = minuit.fval
    prob_fit = stats.chi2.cdf(chi2_fit, ndof_fit)

    # ------------ FIGURE ------------
    fig, ax = plt.subplots(figsize=(20, 8))

    arguments = inspect.getfullargspec(model).args
    parameter_names = arguments[1:]

    value_dict = dict(zip(parameter_names, minuit.values))
    error_dict = dict(zip(parameter_names, minuit.errors))

    d = {'chi2:': chi2_fit,
         'ndf:': ndof_fit,
         'Prob:': prob_fit}

    for name, value in value_dict.items():
        globals()[name + '_fit'] = value
        d[name] = [value]
    for name, error in error_dict.items():
        globals()[name + '_err'] = error
        d[name].append(error)

    string0 = f"Chi2: {d['chi2:']:.4f}\nDegrees of freedom: {d['ndf:']:.0f}\nPropability: {d['Prob:']:.4f}" 
    ax.text(min(x)*text_x, max(y)*text_y, string0, family='monospace', fontsize=12, verticalalignment='top', horizontalalignment='left')
    
    ax.errorbar(xs, ys, yerr=sy, xerr=sx, label='Data with errorbars', fmt='.k', ecolor='k', elinewidth=1, capsize=1, capthick=1)

    mod = model(x_linspace, *minuit.values) 
    ax.plot(x_linspace, mod, color='r', label=fit_label)
    residuals = ys - model(xs, *minuit.values)

    ax.legend()
    ax.set_xlabel(xlabel, fontsize=fontsize)
    ax.set_ylabel(ylabel, fontsize=fontsize)
    ax.set_title(title)
    plt.grid()
    return d, residuals, mod, x_linspace, minuit

def FIT(x, model, initial_params, more_bins , fit_label, xlabel, title, text_x=0.05, text_y=0.95):
    # ------------ HISTOGRAM ------------
    bins = int(np.sqrt(len(x)) * more_bins)
    counts, bin_edges = np.histogram(x, bins=bins)

    xs = 0.5 * (bin_edges[1:][counts > 0] + bin_edges[:-1][counts > 0])
    ys = counts[counts > 0]
    sy = np.sqrt(counts[counts > 0])

    x_linspace = np.linspace(xs.min(), xs.max(), 1000)

    # ------------ FIT ------------
    minuit = chi2_minimization(model, xs, ys, y_err=sy, initial_params=initial_params)

    npara = len(initial_params)
    ndof_fit = len(xs) - npara

    chi2_fit = minuit.fval
    prob_fit = stats.chi2.cdf(chi2_fit, ndof_fit) 

    # ------------ FIGURE ------------
    fig, ax = plt.subplots(figsize=(15, 6))

    arguments = inspect.getfullargspec(model).args
    parameter_names = arguments[1:]

    value_dict = dict(zip(parameter_names, minuit.values))
    error_dict = dict(zip(parameter_names, minuit.errors))

    d = {'chi2:': chi2_fit,
         'ndf:': ndof_fit,
         'Prob:': prob_fit}

    for name, value in value_dict.items():
        globals()[name + '_fit'] = value
        d[name] = [value]
    for name, error in error_dict.items():
        globals()[name + '_err'] = error
        d[name].append(error)

    string0 = f"Chi2: {d['chi2:']:.4f}\nDegrees of freedom: {d['ndf:']:.0f}\nPropability: {d['Prob:']:.4f}" 

    ax.text(min(x), max(counts), string0, family='monospace', fontsize=12, verticalalignment='top', horizontalalignment='left')
    ax.errorbar(xs, ys, yerr=sy, fmt='.k', ecolor='k', elinewidth=1, capsize=1, capthick=1, label = "Data with Poisson errors")
    ax.hist(x, bins=bins, histtype='step', label='Histogram')

    ax.plot(x_linspace, model(x_linspace, *minuit.values), color='r', label=fit_label)

    ax.legend(loc='upper right')
    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel('Frequency', fontsize=15)
    ax.set_title(title)
    plt.grid()

    return d, value_dict, error_dict, minuit



### Data import

#### ERA5

In [ ]:
file_path_ERA5  = "Data/ERA5/ERA5_T2m_2026.nc"
ds_ERA5 = xr.open_dataset(file_path_ERA5)
temp_ERA5_ng = ds_ERA5['t2m'].sel(latitude=slice(55, 45),longitude=slice(10, 30)).mean(dim=["latitude", "longitude"]) - 273.15
temp_ERA5 = temp_ERA5_ng.groupby("valid_time.month")
ds_ERA5 = 0

#### Satelite SIC

In [ ]:
monthly_list = []
monthly_list1 = []
monthly_list2 = []
monthly_list3 = []

files = sorted(glob("Data/SIC/*.nc"))

ref = xr.open_dataset(files[0])
x = ref["x"].values
y = ref["y"].values
xx, yy = np.meshgrid(x, y)

transformer = Transformer.from_crs("EPSG:3411", "EPSG:4326", always_xy=True)
lon2d, lat2d = transformer.transform(xx, yy)

mask = ((lat2d >= 65) & (lat2d <= 80) & (lon2d >= 30) & (lon2d <= 80))
mask_ex = ((lat2d >= 65) & (lat2d <= 90) & (lon2d >= 20) & (lon2d <= 100))

mask_np = ((lat2d >= 70) & (lat2d <= 90) & (lon2d >= 0) & (lon2d <= 360))  
mask_np_l = ((lat2d >= 60) & (lat2d <= 90) & (lon2d >= 0) & (lon2d <= 360)) 

for f in tqdm(files):
    ds = xr.open_dataset(f).drop_vars(
        ["crs","cdr_seaice_conc_qa_flag",
         "cdr_seaice_conc_interp_temporal_flag",
         "cdr_seaice_conc_interp_spatial_flag"],
        errors="ignore"
    )
    sic = ds["cdr_seaice_conc"]
    masked_sic0 = sic.where(mask)
    masked_sic1 = sic.where(mask_ex)
    masked_sic2 = sic.where(mask_np)
    masked_sic3 = sic.where(mask_np_l)

    sic_bk_monthly = masked_sic0.resample(time="1ME").mean(dim=["time", "x", "y"], skipna=True)
    sic_bke_monthly = masked_sic1.resample(time="1ME").mean(dim=["time", "x", "y"], skipna=True)
    sic_np_monthly = masked_sic2.resample(time="1ME").mean(dim=["time", "x", "y"], skipna=True)
    sic_npl_monthly = masked_sic3.resample(time="1ME").mean(dim=["time", "x", "y"], skipna=True)

    monthly_list.append(sic_bk_monthly)
    monthly_list1.append(sic_bke_monthly)
    monthly_list2.append(sic_np_monthly)
    monthly_list3.append(sic_npl_monthly)

SIC_BK_SAT = xr.concat(monthly_list, dim="time")
SIC_BKE_SAT = xr.concat(monthly_list1, dim="time")
SIC_NP_SAT = xr.concat(monthly_list2, dim="time")
SIC_NPL_SAT = xr.concat(monthly_list3, dim="time")

#### EC-Earth3

In [ ]:
file_path_sic  = "Data/LR/r1/sic/sic_t_r1.nc"
ds_sic = xr.open_dataset(file_path_sic)
lon2d = ds_sic['longitude']  
lat2d = ds_sic['latitude']   

mask = ((lat2d >= 65) & (lat2d <= 80) & (lon2d >= 30) & (lon2d <= 80))
mask_ex = ((lat2d >= 65) & (lat2d <= 90) & (lon2d >= 20) & (lon2d <= 100))

mask_np_l = ((lat2d >= 60) & (lat2d <= 90) & (lon2d >= 0) & (lon2d <= 360)) 
mask_np = ((lat2d >= 70) & (lat2d <= 90) & (lon2d >= 0) & (lon2d <= 360)) 

mask0_int = mask.astype(int)
mask1_int = mask_ex.astype(int)
mask2_int = mask_np.astype(int)
mask3_int = mask_np_l.astype(int)

ens_members = ['r1','r4','r6','r9','r11','r13','r15']

ens_members_sr = arr = [f"r{num}" for num in range(101, 151)]


chunk_dict = {'time': 'auto'} 

def process_member(base_path, member):
    """Processes a single ensemble member lazily."""
    file_path_ts   = os.path.join(base_path, member, "ts" , f"ts_t_{member}.nc")
    file_path_sic  = os.path.join(base_path, member, "sic", f"sic_t_{member}.nc")
    file_path_hfls = os.path.join(base_path, member, f"hfls_t_{member}.nc")
    file_path_zg_f = os.path.join(base_path, member, f"zg_f_{member}.nc")
    file_path_zg_t = os.path.join(base_path, member, f"zg_t_{member}.nc")

    ds_ts   = xr.open_dataset(file_path_ts, chunks=chunk_dict)
    ds_sic  = xr.open_dataset(file_path_sic, chunks=chunk_dict)
    ds_hfls = xr.open_dataset(file_path_hfls, chunks=chunk_dict)
    ds_zg_f = xr.open_dataset(file_path_zg_f, chunks=chunk_dict)
    ds_zg_t = xr.open_dataset(file_path_zg_t, chunks=chunk_dict)
    
    subset_hfls = ds_hfls["hfls"].sel(lat=slice(65, 80), lon=slice(30, 80)).mean(dim=["lat", "lon"])
    subset_hfls = subset_hfls.assign_coords(member=member)
    
    subset_temp = ds_ts["tas"].sel(lat=slice(45, 55), lon=slice(10, 30)).mean(dim=["lat", "lon"])
    subset_temp = subset_temp - 273.15 
    subset_temp = subset_temp.assign_coords(member=member)
    
    sic = ds_sic["siconc"]
    
    valid_data_mask = sic.notnull().astype(int)
    
    def fast_masked_mean(mask_int):
        sic_sum = (sic * mask_int).sum(dim=["j", "i"], skipna=True)
        valid_cells = (mask_int * valid_data_mask).sum(dim=["j", "i"])
        return sic_sum / valid_cells

    bk_sic = fast_masked_mean(mask0_int).assign_coords(member=member)
    bke_sic = fast_masked_mean(mask1_int).assign_coords(member=member)
    np_sic = fast_masked_mean(mask2_int).assign_coords(member=member)
    npl_sic = fast_masked_mean(mask3_int).assign_coords(member=member)

    zg_eu = xr.concat([ds_zg_f['zg'].sel(lat=slice(45, 55), lon=slice(10, 30)).mean(dim=["lat", "lon"]), ds_zg_t['zg'].sel(lat=slice(45, 55), lon=slice(10, 30)).mean(dim=["lat", "lon"])], dim="time").sortby("time")
    zg_np = xr.concat([ds_zg_f['zg'].sel(lat=slice(60, 84), lon=slice(300, 350)).mean(dim=["lat", "lon"]), ds_zg_t['zg'].sel(lat=slice(60, 84), lon=slice(300, 350)).mean(dim=["lat", "lon"])], dim="time").sortby("time")
    zg_bk = xr.concat([ds_zg_f['zg'].sel(lat=slice(65, 80), lon=slice(30, 80)).mean(dim=["lat", "lon"]), ds_zg_t['zg'].sel(lat=slice(65, 80), lon=slice(30, 80)).mean(dim=["lat", "lon"])], dim="time").sortby("time")

    zg_eu = zg_eu.assign_coords(member=member)
    zg_np = zg_np.assign_coords(member=member)
    zg_bk = zg_bk.assign_coords(member=member)

    
    return (
            subset_temp.compute(), 
            bk_sic.compute(), 
            subset_hfls.compute(), 
            bke_sic.compute(), 
            np_sic.compute(), 
            npl_sic.compute(),
            zg_eu.compute(),
            zg_np.compute(),
            zg_bk.compute()
        )
LR_path = "Data/LR"
lr_results = [process_member(LR_path, m) for m in tqdm(ens_members, desc="Setting up LR")]

europe_temp_lr, BK_SIC_lr, BK_HFLS_lr, BKE_SIC_lr, NP_SIC_lr, NPL_SIC_lr, ZG_EU_lr, ZG_NP_lr, ZG_BK_lr = [xr.concat(vars, dim="member") for vars in zip(*lr_results)]

SR_path = "Data/SR"
sr_results = [process_member(SR_path, m) for m in tqdm(ens_members_sr, desc="Setting up SR")]

europe_temp_sr, BK_SIC_sr, BK_HFLS_sr, BKE_SIC_sr, NP_SIC_sr, NPL_SIC_sr, ZG_EU_sr, ZG_NP_sr, ZG_BK_sr = [xr.concat(vars, dim="member") for vars in zip(*sr_results)]

print("Concatenating the computed results...")
europe_temp = xr.concat([europe_temp_lr.isel(time=slice(1, None)), europe_temp_sr], dim="member")
BK_SIC      = xr.concat([BK_SIC_lr, BK_SIC_sr], dim="member")
BK_HFLS     = xr.concat([BK_HFLS_lr, BK_HFLS_sr], dim="member")
BKE_SIC     = xr.concat([BKE_SIC_lr, BKE_SIC_sr], dim="member")
NP_SIC      = xr.concat([NP_SIC_lr, NP_SIC_sr], dim="member")
NPL_SIC     = xr.concat([NPL_SIC_lr, NPL_SIC_sr], dim="member")
ZG_EU       = xr.concat([ZG_EU_lr, ZG_EU_sr], dim="member")
ZG_NP       = xr.concat([ZG_NP_lr, ZG_NP_sr], dim="member")
ZG_BK       = xr.concat([ZG_BK_lr, ZG_BK_sr], dim="member")

print("Done!")

In [ ]:
europe_temp_lr = xr.concat(europe_temp_list, dim="member")
europe_temp_lr = europe_temp_lr.isel(time=slice(1, None)) # r11 - r15 has a datapoint for 1849 dec :)))) Why? who knows, nothing else in their data has this (LLNL download IIRC)

BK_SIC_lr = xr.concat(BK_SIC_list, dim="member")
BK_HFLS_lr = xr.concat(BK_HFLS_list, dim="member")

BKE_SIC_lr = xr.concat(BKE_SIC_list, dim="member")
NP_SIC_lr = xr.concat(NP_SIC_list, dim="member")
NPL_SIC_lr = xr.concat(NPL_SIC_list, dim="member")

europe_temp_sr = xr.concat(europe_temp_list_sr, dim="member")

BK_SIC_sr = xr.concat(BK_SIC_list_sr, dim="member")
BK_HFLS_sr = xr.concat(BK_HFLS_list_sr, dim="member")
BKE_SIC_sr = xr.concat(BKE_SIC_list_sr, dim="member")
NP_SIC_sr = xr.concat(NP_SIC_list_sr, dim="member")
NPL_SIC_sr = xr.concat(NPL_SIC_list_sr, dim="member")

europe_temp = xr.concat([europe_temp_lr, europe_temp_sr], dim="member")
BK_SIC = xr.concat([BK_SIC_lr, BK_SIC_sr], dim="member")
BK_HFLS= xr.concat([BK_HFLS_lr, BK_HFLS_sr], dim="member")
BKE_SIC = xr.concat([BKE_SIC_lr, BKE_SIC_sr], dim="member")
NP_SIC = xr.concat([NP_SIC_lr, NP_SIC_sr], dim="member")
NPL_SIC = xr.concat([NPL_SIC_lr, NPL_SIC_sr], dim="member")


##### Save

In [ ]:
datasets = {
    "europe_temp": europe_temp,
    "BK_SIC": BK_SIC,
    "BK_HFLS": BK_HFLS,
    "BKE_SIC": BKE_SIC,
    "NP_SIC": NP_SIC,
    "NPL_SIC": NPL_SIC,
    "ZG_EU": ZG_EU,
    "ZG_NP": ZG_NP,
    "ZG_BK": ZG_BK
}

for name, ds in datasets.items():
    file_name = f"{name}.nc"
    ds.to_netcdf(file_name)
    
    print(f"Saved {file_name} with shape {ds.shape}")

#### Load EC-EARTH3

In [ ]:
europe_temp = xr.open_dataarray("europe_temp.nc")
BK_SIC      = xr.open_dataarray("BK_SIC.nc")
NP_SIC      = xr.open_dataarray("NP_SIC.nc")
BK_HFLS     = xr.open_dataarray("BK_HFLS.nc")
BKE_SIC     = xr.open_dataarray("BKE_SIC.nc")
NP_SIC      = xr.open_dataarray("NP_SIC.nc")
NPL_SIC     = xr.open_dataarray("NPL_SIC.nc")
ZG_EU       = xr.open_dataarray("ZG_EU.nc").squeeze()
ZG_NP       = xr.open_dataarray("ZG_NP.nc").squeeze()
ZG_BK       = xr.open_dataarray("ZG_BK.nc").squeeze()


### Cold Extreme over time

In [ ]:
europe_temp_clim = europe_temp.sel(time=slice("1971-01", "2000-12"))

combined_member_std = europe_temp_clim.groupby("time.month").std(dim="time")
combined_member_mean = europe_temp_clim.groupby("time.month").mean(dim="time")

climate_std_within_members = combined_member_std.mean(dim="member")
climate_normal = combined_member_mean.mean(dim="member")  


months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for i in range(1,13):  
    month_name = months[i - 1]

    mean_val = climate_normal.sel(month=i)
    std_val = climate_std_within_members.sel(month=i)

    print(f"The average temperature for {month_name} is: {mean_val.values:5.2f} ± {std_val.values:.2f} °C")


#### Ensemble comparison (normal)

In [ ]:
time_slices = [["1971-01", "2000-12"] , ["1970-01", "2005-12"] , ["2006-01", "2050-12"], ["2051-01", "2100-12"]]

ncold = xr.zeros_like(combined_member_mean).rename('ncold')
nwarm = xr.zeros_like(combined_member_mean).rename('nwarm')

def gaussian_fit(x, N, mean, std):
    return N * 1/(std * np.sqrt(2 * np.pi)) * np.exp(-0.5 * (x-mean)**2 / std**2)

what_to_plot = 0 

for k in range(4):
    eu = europe_temp.sel(time=slice(time_slices[k][0], time_slices[k][1]))
    member_temp = eu.groupby("time.month").mean(dim="time")
    member_mean = member_temp.mean(dim="member")

    for i in range(1,13):
        diftomean = member_temp.sel(month = i) - member_mean.sel(month = i)

        top_n = 10
        
        smallest_vals = np.sort(diftomean.values)[:top_n]
        smallest_idx = np.argsort(diftomean.values)[:top_n]
        
        largest_vals = np.sort(diftomean.values)[-top_n:]
        largest_idx = np.argsort(diftomean.values)[-top_n:]
        
        mask_c = np.zeros_like(diftomean.values, dtype=bool)
        mask_c[smallest_idx] = True
        
        mask_w = np.zeros_like(diftomean.values, dtype=bool)
        mask_w[largest_idx] = True
        
        ncold.loc[dict(month=i)] = ncold.sel(month=i) + mask_c.astype(int)
        nwarm.loc[dict(month=i)] = nwarm.sel(month=i) + mask_w.astype(int)
        
        if k == what_to_plot:
            fit_label = 'Gaussian fit'
            title = f'Ensemble distribution around mean temperature in {months[i-1]} in range {time_slices[k]}'
            xlabel = 'Temperature difference to mean [°C]'
            initial_params = [16 , 0 , 1]

            FIT(diftomean, gaussian_fit, initial_params, 1.5, fit_label, xlabel, title, text_x=0.05, text_y=0.95)
        
            plt.text(np.min(diftomean), 0.75, str(diftomean[diftomean < np.min(diftomean)*0.99].member.values))
            plt.text(np.max(diftomean), 0.75, str(diftomean[diftomean > np.max(diftomean)*0.99].member.values))
            
    
    if k == 0:                                                      # Climate normal time span     ("1990-01", "2014-12")
        ncold_no = ncold.copy()
        nwarm_no = nwarm.copy()
    
    elif k == 1:                                                    # Climate present time span    ("2015-01", "2039-12")
        ncold_pr = (ncold - ncold_no).copy()
        nwarm_pr = (nwarm - nwarm_no).copy()
    
    elif k == 2:                                                    # Climate future time span     ("2040-01", "2069-12")
        ncold_fu = (ncold - ncold_no - ncold_pr).copy()
        nwarm_fu = (nwarm - nwarm_no - nwarm_pr).copy()
    
    elif k == 3:                                                    # Climate far future time span ("2070-01", "2100-12")
        ncold_ff = (ncold - ncold_no - ncold_pr - ncold_fu).copy()
        nwarm_ff = (nwarm - nwarm_no - nwarm_pr - nwarm_fu).copy()




In [ ]:
ncold_ext = [ncold_no, ncold_pr, ncold_fu, ncold_ff, ncold]
nwarm_ext = [nwarm_no, nwarm_pr, nwarm_fu, nwarm_ff, nwarm]
time_slices = [["1971-01", "2000-12"] , ["1970-01", "2005-12"] , ["2006-01", "2050-12"], ["2051-01", "2100-12"], ["1990-01", "2100-12"]]

surv_cold =  []
surv_warm =  []

for i in range(len(ncold_ext)):
    S = ncold_ext[i].sum(dim="month")  
    
    sorted_vals = np.sort(S.values)[::-1]
    
    score = sorted_vals[4]
    score_max = sorted_vals[0]

    survivors_c = S[S >= score].member.values
    surv_cold.append(survivors_c)
    print(f"For extreme cold score ({time_slices[i]}): {score_max:.0f}-{score:.0f}\n", survivors_c)
print('\n')

for i in range(len(nwarm_ext)):
    S = nwarm_ext[i].sum(dim="month")  
    
    sorted_vals = np.sort(S.values)[::-1]
    
    score = sorted_vals[4]
    score_max = sorted_vals[0]
    
    survivors_w = S[S >= score].member.values
    surv_warm.append(survivors_w)

    print(f"For extreme warm score ({time_slices[i]}): {score_max:.0f}-{score:.0f}\n", survivors_w)

##### Seasonal cycle

In [ ]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
          'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
members_list = (['r1', 'r4', 'r6', 'r9', 'r11', 'r13', 'r15', 'r101', 'r102', 'r103',
       'r104', 'r105', 'r106', 'r107', 'r108', 'r109', 'r110', 'r111', 'r112',
       'r113', 'r114', 'r115', 'r116', 'r117', 'r118', 'r119', 'r120', 'r121',
       'r122', 'r123', 'r124', 'r125', 'r126', 'r127', 'r128', 'r129', 'r130',
       'r131', 'r132', 'r133', 'r134', 'r135', 'r136', 'r137', 'r138', 'r139',
       'r140', 'r141', 'r142', 'r143', 'r144', 'r145', 'r146', 'r147', 'r148', 'r149', 'r150'])


month_labels = [m.capitalize() for m in months]

monthly_mean_vals = [] 
monthly_mean_vals_time = []
monthly_mean_vals_diff_t = []
monthly_mean_vals_i_t = [] 

for i in range(1,13):
    monthly_mean_vals.append(europe_temp.sel(time=slice("1971-01", "2005-12")).groupby("time.month")[i].mean(dim = 'member').mean().values)
    monthly_mean_vals_time.append(i)

fig, ax = plt.subplots(nrows= 1,ncols = 2,figsize=(16,8))


for members in members_list:
    monthly_mean_vals_run = []
    for i in range(1,13):
        monthly_mean_vals_run.append(
            europe_temp.sel(time=slice("1971-01", "2005-12"))
            .groupby("time.month")[i]
            .sel(member=members)
            .mean()
            .values)
        
    diff = np.array(monthly_mean_vals_run) - np.array(monthly_mean_vals)
    monthly_mean_vals_diff_t.append(diff)
    monthly_mean_vals_i_t.append(monthly_mean_vals_run)
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_run, c='gray', alpha=0.5)


monthly_mean_vals_diff = xr.DataArray(
    np.array(monthly_mean_vals_diff_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="diff",
)

monthly_mean_vals_i = xr.DataArray(
    np.array(monthly_mean_vals_i_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="val",
)

ax[1].boxplot(monthly_mean_vals_diff, positions = monthly_mean_vals_time);


monthly_mean_vals_diff_sum = monthly_mean_vals_diff.sum(dim='month').values
idx_large = np.argsort(monthly_mean_vals_diff_sum)[-5:]
idx_small = np.argsort(monthly_mean_vals_diff_sum)[:5]

winter = monthly_mean_vals_diff.sel(month=[12,1,2,3]).sum(dim="month")
summer = monthly_mean_vals_diff.sel(month=[6,7,8,9]).sum(dim="month")

mask = (winter > 0) & (summer < 0)
members_dampened_idx = np.argwhere(mask.values).flatten()

for i in range(len(idx_large)):
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i]], alpha = 0.5, color='red')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i]],alpha = 0.5, color='red')
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i]], alpha = 0.5, color='blue')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i]],alpha = 0.5, color='blue')


ax[0].plot(monthly_mean_vals_time, monthly_mean_vals, '-.',marker = 'o', color='k', label='Climate normal mean')
ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[1]], alpha = 0.5, color='red', label='Warm')
ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[1]], alpha = 0.5, color='red', label='Warm')
ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[1]], alpha = 0.5, color='blue', label='Cold')
ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[1]], alpha = 0.5, color='blue', label='Cold')
        

for i in range(2):
    ax[i].set_xticks(ticks=range(1, 13), labels= month_labels, fontsize=14)
    ax[i].set_xlabel("Month")
    ax[i].grid()
    ax[i].legend()

plt.suptitle("Seasonal cycle (1971–2005 baseline)", fontsize=16)

ax[0].set_title("Temperature", fontsize=18)
ax[0].set_title("Temperature difference to mean", fontsize=18)

ax[0].set_ylabel("European temperature (°C)", fontsize=18)
ax[1].set_ylabel("Temp difference (°C)", fontsize=18)
fig.savefig("SeasonalCycle.png", bbox_inches="tight", dpi=300)

#### Ensemble over time

In [ ]:
print("The 5 Coldest are: ",europe_temp[idx_small].member.values)
print("The 5 Warmest are: ",europe_temp[idx_large].member.values)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll = 60

array_list = [members_list,surv_cold[0],surv_warm[0]]
array_list = [members_list,europe_temp[idx_small].member.values,europe_temp[idx_large].member.values]

label_list = ['Normal','Cold in normal','Warm in normal']
color_list = ['gray','blue','red']

for i in range(len(array_list)):
    members_list_temp = array_list[i]
    for members in members_list_temp:
        y = europe_temp.sel(member = members)
        x = europe_temp.sel(member = members).time
            
        y_roll = y.rolling(time = time_roll, center=False).mean()
        axes.plot(x , y_roll, c = color_list[i], alpha = 0.5)


normal_line = mlines.Line2D([], [], color='gray', alpha=0.5, label='Ensemble members')
cold_line   = mlines.Line2D([], [], color='blue', alpha=0.5, label='Cold in normal')
warm_line   = mlines.Line2D([], [], color='red',  alpha=0.5, label='Warm in normal')
axes.set(title = 'Running mean temperature over time', ylabel = 'Temperature [°C]', xlabel = 'Time', xlim=(0, 48500))
axes.legend(handles=[normal_line, cold_line, warm_line])
axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
axes.grid()
# fig.savefig("RunMeanTemp.png", bbox_inches="tight", dpi=100)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll = 60

normal_members = members_list
cold_members   = europe_temp[idx_small].member.values
warm_members   = europe_temp[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

# Normal members (gray)
for m in normal_members:
    y = europe_temp.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c='gray', alpha=0.3)

# Cold (shades of blue)
for m, col in zip(cold_members, cold_colors):
    y = europe_temp.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

# Warm (shades of red)
for m, col in zip(warm_members, warm_colors):
    y = europe_temp.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

# Real data (black)
era5_roll = temp_ERA5_ng.rolling(valid_time=time_roll, center=False).mean()
axes.plot(era5_roll.valid_time , era5_roll, c = 'black', alpha = 1)

axes.set(
    title='5-Year Running mean temperature over time',
    ylabel='Temperature [°C]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.02
y0 = 0.95
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')
axes.text(x0, y0 - dy, "Real data", transform=axes.transAxes,
          color='black', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')
    
axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
fig.savefig("RunMeanTemp.png", bbox_inches="tight", dpi=50)

In [ ]:

fig, axes = plt.subplots(1, 1, figsize=(14, 10))
diffs = []       
members_all = [] 

for i in range(len(array_list)):
    members_list_temp = array_list[i]
    for members in members_list_temp:
        y = BK_SIC.sel(member = members)
        x = BK_SIC.sel(member = members).time
    
        y_roll = y.rolling(time = time_roll, center=False).mean()
        axes.plot(x , y_roll, c = color_list[i], alpha = 0.45)
        if i == 0:
            diff_sum = np.sum(y.sel(time=slice("1980-01", "2024-12")).values 
            - SIC_BK_SAT.sel(time=slice("1980-01", "2024-12")).values * 100)
                    
            diffs.append(diff_sum)

axes.set(title = 'Running mean BK SIC over time', ylabel = 'BK SIC [%]', xlabel = 'Time', xlim=(0, 48500))
axes.plot(SIC_BK_SAT.time , SIC_BK_SAT.rolling(time = time_roll, center=False).mean() * 100, c = 'black', alpha = 1)

normal_line = mlines.Line2D([], [], color='gray', alpha=0.5, label='Ensemble member')
cold_line   = mlines.Line2D([], [], color='blue', alpha=0.5, label='Cold in normal')
warm_line   = mlines.Line2D([], [], color='red',  alpha=0.5, label='Warm in normal')
real_line   = mlines.Line2D([], [], color='black',  alpha=1, label='Real BK data')

axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
axes.grid()
axes.legend(handles=[normal_line, cold_line, warm_line,real_line])
# fig.savefig("RunMeanBKSIC.png", bbox_inches="tight", dpi=100)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll = 60

normal_members = members_list
cold_members   = BK_SIC[idx_small].member.values
warm_members   = BK_SIC[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

# Normal members (gray)
for m in normal_members:
    y = BK_SIC.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c='gray', alpha=0.4)

# Cold (shades of blue)
for m, col in zip(cold_members, cold_colors):
    y = BK_SIC.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

# Warm (shades of red)
for m, col in zip(warm_members, warm_colors):
    y = BK_SIC.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

axes.plot(SIC_BK_SAT.time , SIC_BK_SAT.rolling(time = time_roll, center=False).mean() * 100, c = 'black', alpha = 1)


axes.set(
    title='5-Year Running mean BK SIC over time',
    ylabel= 'BK SIC [%]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.93
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')
axes.text(x0, y0 - dy, "Real data", transform=axes.transAxes,
          color='black', fontsize=14, va='top')
axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanBKSIC.png", bbox_inches="tight", dpi=100)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll = 60

normal_members = members_list
cold_members   = BK_HFLS[idx_small].member.values
warm_members   = BK_HFLS[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))
for m in normal_members:
    y = BK_HFLS.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = BK_HFLS.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = BK_HFLS.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)



axes.set(
    title='Running mean Latent Heat Flux over time',
    ylabel= 'Latent Heat Flux [W / m^2]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')
axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanBKHFLS.png", bbox_inches="tight", dpi=100)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll = 60

normal_members = members_list
cold_members   = ZG_BK[idx_small].member.values
warm_members   = ZG_BK[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

for m in normal_members:
    y = ZG_BK.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = ZG_BK.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = ZG_BK.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)


axes.set(
    title='5-Year Running mean Geopotential Height in BK over time',
    ylabel= 'Geopotential Height [m]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanBKZG.png", bbox_inches="tight", dpi=100)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll = 60

normal_members = members_list
cold_members   = ZG_NP[idx_small].member.values
warm_members   = ZG_NP[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

for m in normal_members:
    y = ZG_NP.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = ZG_NP.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = ZG_NP.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)



axes.set(
    title='5-Year Running mean Geopotential Height in NP over time',
    ylabel= 'Geopotential Height [m]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')
axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
fig.savefig("RunMeanNPZG.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll = 60

normal_members = members_list
cold_members   = ZG_EU[idx_small].member.values
warm_members   = ZG_EU[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))
for m in normal_members:
    y = ZG_EU.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = ZG_EU.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = ZG_EU.sel(member=m)
    x = y.time
    y_roll = y.rolling(time=time_roll, center=False).mean()
    axes.plot(x, y_roll, c=col, alpha=0.8)



axes.set(
    title='5-Year Running mean Geopotential Height in EU over time',
    ylabel= 'Geopotential Height [m]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanEUZG.png", bbox_inches="tight", dpi=100)

#### Ensemble over time DJF

##### SIC

In [ ]:
def get_djf_rolling(da, window):
    djf = da.sel(time=da.time.dt.month.isin([12, 1, 2]))
    return djf.resample(time='YS-DEC').mean().rolling(time=window, center=False).mean()


In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))
def get_djf_rolling(da, window):
    djf = da.sel(time=da.time.dt.month.isin([12, 1, 2]))
    return djf.resample(time='YS-DEC').mean().rolling(time=window, center=False).mean()

time_roll_years = 5

normal_members = members_list
cold_members   = BK_SIC[idx_small].member.values
warm_members   = BK_SIC[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))
for m in normal_members:
    y = BK_SIC.sel(member=m)
    x = y.time
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = BK_SIC.sel(member=m)
    x = y.time
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = BK_SIC.sel(member=m)
    x = y.time
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

real_roll = get_djf_rolling(SIC_BK_SAT,time_roll_years)
axes.plot(real_roll.time , real_roll * 100, c = 'black', alpha = 1)


axes.set(
    title='5-Year Running mean DJF BK SIC over time',
    ylabel= 'BK SIC [%]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.93
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')
axes.text(x0, y0 - dy, "Real data", transform=axes.transAxes,
          color='black', fontsize=14, va='top')
axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanBKSIC_DJF.png", bbox_inches="tight", dpi=100)

##### TAS

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll_years = 5

normal_members = members_list
cold_members   = europe_temp[idx_small].member.values
warm_members   = europe_temp[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

def get_djf_rolling_era5(da, window):
    djf = da.sel(valid_time=da.valid_time.dt.month.isin([12, 1, 2]))
    y_roll = (djf.resample(valid_time='YS-DEC').mean().rolling(valid_time=window, center=False).mean())
    return y_roll
    
for m in normal_members:
    y = europe_temp.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c='gray', alpha=0.3)

for m, col in zip(cold_members, cold_colors):
    y = europe_temp.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = europe_temp.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

era5_roll = get_djf_rolling_era5(temp_ERA5_ng, 5)
axes.plot(era5_roll.valid_time , era5_roll, c = 'black', alpha = 1)


axes.set(
    title='5-Year Running Mean of DJF Temperature',
    ylabel='Temperature [°C]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.02
y0 = 0.95
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')
axes.text(x0, y0 - dy, "Real data", transform=axes.transAxes,
          color='black', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')
axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanTemp_DJF.png", bbox_inches="tight", dpi=100)

##### HFLS

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll_years = 5

normal_members = members_list
cold_members   = BK_HFLS[idx_small].member.values
warm_members   = BK_HFLS[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

for m in normal_members:
    y = BK_HFLS.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = BK_HFLS.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = BK_HFLS.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)


axes.set(
    title='5-Year Running mean DJF Latent Heat Flux sum over time',
    ylabel= 'Latent Heat Flux sum [W / m^2]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanBKHFLS_DJF.png", bbox_inches="tight", dpi=100)

##### ZG BK

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll_years = 5

normal_members = members_list
cold_members   = ZG_BK[idx_small].member.values
warm_members   = ZG_BK[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))
for m in normal_members:
    y = ZG_BK.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = ZG_BK.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = ZG_BK.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)



axes.set(
    title='5-Year Running mean DJF Geopotential Height in BK over time',
    ylabel= 'Geopotential Height[m]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
fig.savefig("RunMeanBKZG_DJF.png", bbox_inches="tight", dpi=300)

##### ZG NP

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll_years = 5

normal_members = members_list
cold_members   = ZG_NP[idx_small].member.values
warm_members   = ZG_NP[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

for m in normal_members:
    y = ZG_NP.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = ZG_NP.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = ZG_NP.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)



axes.set(
    title='5-Year Running mean DJF Geopotential Height in NP over time',
    ylabel= 'Geopotential Height [m]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
fig.savefig("RunMeanNPZG_DJF.png", bbox_inches="tight", dpi=300)

##### ZG EU

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))

time_roll_years = 5

normal_members = members_list
cold_members   = ZG_EU[idx_small].member.values
warm_members   = ZG_EU[idx_large].member.values

cold_colors = cm.Blues(np.linspace(1, 0.5, len(cold_members)))
warm_colors = cm.Reds(np.linspace(1.0, 0.5, len(warm_members)))

for m in normal_members:
    y = ZG_EU.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c='gray', alpha=0.4)

for m, col in zip(cold_members, cold_colors):
    y = ZG_EU.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)

for m, col in zip(warm_members, warm_colors):
    y = ZG_EU.sel(member=m)
    y_roll = get_djf_rolling(y, time_roll_years)
    axes.plot(y_roll.time, y_roll, c=col, alpha=0.8)



axes.set(
    title='5-Year Running mean DJF Geopotential Height in EU over time',
    ylabel= 'Geopotential Height [m]',
    xlabel='Time',
    xlim=(0, 48500)
)
axes.grid()

x0 = 0.82
y0 = 0.53
dy = 0.035

axes.text(x0, y0, "Normal members", transform=axes.transAxes,
          color='gray', fontsize=14, va='top')

axes.text(x0, y0 - 2*dy, "Cold members:", transform=axes.transAxes,
          color='blue', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(cold_members, cold_colors)):
    axes.text(x0 + 0.02, y0 - dy - dy*(i+2), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')

offset = y0 - dy*(len(cold_members)+3)
axes.text(x0, offset, "Warm members:", transform=axes.transAxes,
          color='red', fontsize=14, va='top')

for i, (m, col) in enumerate(zip(warm_members, warm_colors)):
    axes.text(x0 + 0.02, offset - dy*(i+1), str(m),
              transform=axes.transAxes, color=col, fontsize=12, va='top')


axes.title.set_fontsize(20)
axes.xaxis.label.set_size(16)
axes.yaxis.label.set_size(16)
# fig.savefig("RunMeanEUZG_DJF.png", bbox_inches="tight", dpi=100)

#### Ensemble extremes over time

In [ ]:
def calc_historical_zscore(month_data, window=20):
    past_data = month_data.shift(time=1)
    
    rolled = past_data.rolling(time=window).construct("window_dim")
    
    baseline_mean = rolled.mean(dim=["member", "window_dim"], skipna=True)
    baseline_std = rolled.std(dim=["member", "window_dim"], skipna=True)
    
    return (month_data - baseline_mean) / baseline_std

eu_zscores = (
    europe_temp.groupby("time.month")
    .map(calc_historical_zscore)
    .sortby("time")
)


#### Z-scores 

In [ ]:
def calc_historical_zscore(month_data, window=20):
    past_data = month_data.shift(time=1)
    
    rolled = past_data.rolling(time=window).construct("window_dim")
    
    baseline_mean = rolled.mean(dim=["member", "window_dim"], skipna=True)
    baseline_std = rolled.std(dim=["member", "window_dim"], skipna=True)
    
    return (month_data - baseline_mean) / baseline_std

eu_zscores = (europe_temp.groupby("time.month").map(calc_historical_zscore).sortby("time"))

bk_hfls_zscores = (BK_HFLS.groupby("time.month").map(calc_historical_zscore).sortby("time"))

bk_sic_zscores = (BK_SIC.groupby("time.month").map(calc_historical_zscore).sortby("time"))

np_sic_zscores = (NPL_SIC.groupby("time.month").map(calc_historical_zscore).sortby("time"))

bk_zg_zscores = (ZG_BK.groupby("time.month").map(calc_historical_zscore).sortby("time"))

eu_zg_zscores = (ZG_EU.groupby("time.month").map(calc_historical_zscore).sortby("time"))

np_zg_zscores = (ZG_NP.groupby("time.month").map(calc_historical_zscore).sortby("time"))


#### Real extremes

In [ ]:
djf_ERA5_d = temp_ERA5_ng.where(temp_ERA5_ng['valid_time.month'].isin([12,1,2]), drop=True)

djf_ERA5_d = temp_ERA5_ng.where(temp_ERA5_ng['valid_time.month'].isin([12]), drop=True)
djf_ERA5_j = temp_ERA5_ng.where(temp_ERA5_ng['valid_time.month'].isin([1]), drop=True)
djf_ERA5_f = temp_ERA5_ng.where(temp_ERA5_ng['valid_time.month'].isin([2]), drop=True)

era5_clim_d = djf_ERA5_d.sel(valid_time=slice("1971-01-01", "2000-12-31")).mean()
era5_clim_j = djf_ERA5_j.sel(valid_time=slice("1971-01-01", "2000-12-31")).mean()
era5_clim_f = djf_ERA5_f.sel(valid_time=slice("1971-01-01", "2000-12-31")).mean()

era5_cli_d = (era5_clim_d.values > djf_ERA5_d.sel(valid_time=slice("1971-01-01", "2000-12-31")))
era5_cli_j = (era5_clim_j.values > djf_ERA5_j.sel(valid_time=slice("1971-01-01", "2000-12-31")))
era5_cli_f = (era5_clim_f.values > djf_ERA5_f.sel(valid_time=slice("1971-01-01", "2000-12-31")))

era5_hist_d = (era5_clim_d.values > djf_ERA5_d.sel(valid_time=slice("1970-01-01", "2005-12-31")))
era5_hist_j = (era5_clim_j.values > djf_ERA5_j.sel(valid_time=slice("1970-01-01", "2005-12-31")))
era5_hist_f = (era5_clim_f.values > djf_ERA5_f.sel(valid_time=slice("1970-01-01", "2005-12-31")))

era5_fut_d = (era5_clim_d.values > djf_ERA5_d.sel(valid_time=slice("2006-01-01", "2025-12-31")))
era5_fut_j = (era5_clim_j.values > djf_ERA5_j.sel(valid_time=slice("2006-01-01", "2025-12-31")))
era5_fut_f = (era5_clim_f.values > djf_ERA5_f.sel(valid_time=slice("2006-01-01", "2025-12-31")))

prob_cli_era5 = (era5_cli_d.values.sum() + era5_cli_j.values.sum() + era5_cli_f.values.sum()) / (len(era5_hist_d) + len(era5_hist_j) + len(era5_hist_f))
prob_fut_era5 = (era5_fut_d.values.sum() + era5_fut_j.values.sum() + era5_fut_f.values.sum()) / (len(era5_hist_d) + len(era5_hist_j) + len(era5_hist_f))
prob_hist_era5 = (era5_hist_d.values.sum() + era5_hist_j.values.sum() + era5_hist_f.values.sum()) / (len(era5_hist_d) + len(era5_hist_j) + len(era5_hist_f))

In [ ]:
print("##### ERA5 #####")
print(f"The average temperature for Dec is: {era5_clim_d.values:5.2f} ± {djf_ERA5_d.sel(valid_time=slice("1971-01-01", "2000-12-31")).std().values:.2f} °C")
print(f"The average temperature for Jan is: {era5_clim_j.values:5.2f} ± {djf_ERA5_j.sel(valid_time=slice("1971-01-01", "2000-12-31")).std().values:.2f} °C")
print(f"The average temperature for Feb is: {era5_clim_f.values:5.2f} ± {djf_ERA5_f.sel(valid_time=slice("1971-01-01", "2000-12-31")).std().values:.2f} °C\n")

print("### EC-EARTH3 ###")
for i in (12,1,2):  
    month_name = months[i - 1]

    mean_val = climate_normal.sel(month=i)
    std_val = climate_std_within_members.sel(month=i)

    print(f"The average temperature for {month_name} is: {mean_val.values:5.2f} ± {std_val.values:.2f} °C")

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(16, 8))

ax.plot(djf_ERA5_d.valid_time,(djf_ERA5_d - era5_clim_d)/djf_ERA5_d.sel(valid_time=slice("1971-01-01", "2000-12-31")).std(),'-o')
ax.plot(djf_ERA5_j.valid_time,(djf_ERA5_j - era5_clim_j)/djf_ERA5_j.sel(valid_time=slice("1971-01-01", "2000-12-31")).std(),'-o')
ax.plot(djf_ERA5_f.valid_time,(djf_ERA5_f - era5_clim_f)/djf_ERA5_f.sel(valid_time=slice("1971-01-01", "2000-12-31")).std(),'-o')
ax.grid()

#### Coldest extremes (future)

In [ ]:
europe_temp_fut = europe_temp.sel(time=slice("2006-01", "2100-12"))

BK_SIC_fut = BK_SIC.sel(time=slice("2006-01", "2100-12"))
BKE_SIC_fut = BKE_SIC.sel(time=slice("2006-01", "2100-12"))
NP_SIC_fut = NP_SIC.sel(time=slice("2006-01", "2100-12"))
NPL_SIC_fut = NPL_SIC.sel(time=slice("2006-01", "2100-12"))


monthly_groups = europe_temp_fut.groupby("time.month")
monthly_groups_BK = BK_SIC_fut.groupby("time.month")
monthly_groups_BKE = BKE_SIC_fut.groupby("time.month")
monthly_groups_NP = NP_SIC_fut.groupby("time.month")
monthly_groups_NPL = NPL_SIC_fut.groupby("time.month")


##################################
n1 = 10  # number of smallest values
n = 3  # number for plotting

min_vals = []
min_times = []
min_BK = []
min_BKE = []
min_NP = []
min_NPL = []

max_vals = []
max_times = []
max_BK = []
max_BKE = []
max_NP = []
max_NPL = []


for m in range(1, 13):
    month_data = monthly_groups[m]  
    month_data_BK = monthly_groups_BK[m]  
    month_data_BKE = monthly_groups_BKE[m]  
    month_data_NP = monthly_groups_NP[m]  
    month_data_NPL = monthly_groups_NPL[m] 
    
    sorted_idx = np.argsort(month_data.values, axis=month_data.get_axis_num("time"))

    idx_min = xr.DataArray(sorted_idx[:, :n1], dims=("member", "rank"))
    min_vals.append(month_data.isel(time=idx_min))
    min_BK.append(month_data_BK.isel(time=idx_min))
    min_BKE.append(month_data_BKE.isel(time=idx_min))
    min_NP.append(month_data_NP.isel(time=idx_min))
    min_NPL.append(month_data_NPL.isel(time=idx_min))
    min_times.append(month_data.time.isel(time=idx_min))

    idx_max = xr.DataArray(sorted_idx[:, -n1:], dims=("member", "rank"))
    max_vals.append(month_data.isel(time=idx_max))
    max_BK.append(month_data_BK.isel(time=idx_max))
    max_BKE.append(month_data_BKE.isel(time=idx_max))
    max_NP.append(month_data_NP.isel(time=idx_max))
    max_NPL.append(month_data_NPL.isel(time=idx_max))
    max_times.append(month_data.time.isel(time=idx_max))

min_vals_da = xr.concat(min_vals, dim="month")
min_vals_da["month"] = np.arange(1, 13)

max_vals_da = xr.concat(max_vals, dim="month")
max_vals_da["month"] = np.arange(1, 13)

min_BK_da = xr.concat(min_BK, dim="month")
min_BK_da["month"] = np.arange(1, 13)

max_BK_da = xr.concat(max_BK, dim="month")
max_BK_da["month"] = np.arange(1, 13)

min_BKE_da = xr.concat(min_BKE, dim="month")
min_BKE_da["month"] = np.arange(1, 13)

max_BKE_da = xr.concat(max_BKE, dim="month")
max_BKE_da["month"] = np.arange(1, 13)

min_NP_da = xr.concat(min_NP, dim="month")
min_NP_da["month"] = np.arange(1, 13)

max_NP_da = xr.concat(max_NP, dim="month")
max_NP_da["month"] = np.arange(1, 13)

min_NPL_da = xr.concat(min_NPL, dim="month")
min_NPL_da["month"] = np.arange(1, 13)

max_NPL_da = xr.concat(max_NPL, dim="month")
max_NPL_da["month"] = np.arange(1, 13)

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 14))
axes = axes.flatten()  
norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis
sm = ScalarMappable(norm=norm, cmap=cmap)

for m in range(1, 13):
    ax = axes[m - 1]  

    month_data = monthly_groups[m] 
    month_data_BK = monthly_groups_BK[m]
    climatology = climate_normal.sel(month=m)
    
    true_idx = month_data < climatology

    std = []
    for members in members_list:
        std = (combined_member_std.sel(month = m, member = members))
        temperature_values = month_data.sel(member = members)[true_idx.sel(member = members)]
        BK_val = month_data_BK.sel(member = members)[true_idx.sel(member = members)]

        standardized_anomalies = (temperature_values - climatology ) / std
        ax.scatter(temperature_values.time , standardized_anomalies, marker = 'o' ,c=BK_val, cmap=cmap, norm=norm,)
    
    ax.set(ylabel="Standard deviation [σ]",title = months[m - 1].capitalize() , xlim = (11000,50000))
    ax.grid(True)

plt.suptitle("All CWMs Compared To Own Ensemble σ (1971–2000 climatology)", fontsize=16)

fig.subplots_adjust(right=0.88)  
cbar_ax = fig.add_axes([1.0, 0.15, 0.02, 0.7])  
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, 100, 11))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, 100, 11)])
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

fig.savefig("CWMsSigmaOwn.png", bbox_inches="tight", dpi=300)

In [ ]:
winter_months = [12, 1, 2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis
sm = ScalarMappable(norm=norm, cmap=cmap)

for idx, m in enumerate(winter_months):
    ax = axes[idx]

    month_data = monthly_groups[m]
    month_data_BK = monthly_groups_BK[m]
    climatology = climate_normal.sel(month=m)

    true_idx = month_data < climatology

    for members in members_list:
        std = combined_member_std.sel(month=m, member=members)
        temperature_values = month_data.sel(member=members)[true_idx.sel(member=members)]
        BK_val = month_data_BK.sel(member=members)[true_idx.sel(member=members)]

        standardized_anomalies = (temperature_values - climatology) / std

        ax.scatter(
            temperature_values.time,
            standardized_anomalies,
            marker='o',
            c=BK_val,
            cmap=cmap,
            norm=norm
        )

    ax.set(
        ylabel="Standard deviation [σ]",
        title=months[m - 1].capitalize(),
        xlim=(11000, 50000)
    )
    ax.grid(True)

plt.suptitle("All CWMs Compared To Own Ensemble σ (1971–2000 climatology)", fontsize=16)

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, 100, 11))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, 100, 11)])

plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.show()

fig.savefig("CWMsSigmaOwn_DJF.png", bbox_inches="tight", dpi=300)


In [ ]:

fig, axes = plt.subplots(4, 3, figsize=(12, 14))
axes = axes.flatten()  
norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis
sm = ScalarMappable(norm=norm, cmap=cmap)

for m in range(1, 13):
    ax = axes[m - 1]  

    month_data = monthly_groups[m] 
    month_data_BK = monthly_groups_BK[m]
    climatology = climate_normal.sel(month=m)
    
    true_idx = month_data < climatology

    for members in members_list:
        std = climate_std_within_members.sel(month=m)
        temperature_values = month_data.sel(member = members)[true_idx.sel(member = members)]
        BK_val = month_data_BK.sel(member = members)[true_idx.sel(member = members)]

        standardized_anomalies = (temperature_values - climatology ) / std
        ax.scatter(temperature_values.time , standardized_anomalies, marker = 'o' ,c=BK_val, cmap=cmap, norm=norm,)
    
    ax.set(ylabel="Standard deviation [σ]",title = months[m - 1].capitalize() , xlim = (11000,50000))
    ax.grid(True)

plt.suptitle("All CWMs Compared To Climatology σ (1971–2000 climatology)", fontsize=16)

fig.subplots_adjust(right=0.88)  
cbar_ax = fig.add_axes([1.0, 0.15, 0.02, 0.7])  
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, 100, 11))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, 100, 11)])
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

fig.savefig("CWMsSigmaClim.png", bbox_inches="tight", dpi=300)

In [ ]:
winter_months = [12, 1, 2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis
sm = ScalarMappable(norm=norm, cmap=cmap)

for idx, m in enumerate(winter_months):
    ax = axes[idx]

    month_data = monthly_groups[m]
    month_data_BK = monthly_groups_BK[m]
    climatology = climate_normal.sel(month=m)

    true_idx = month_data < climatology

    for members in members_list:

        std = climate_std_within_members.sel(month=m)
        temperature_values = month_data.sel(member=members)[true_idx.sel(member=members)]
        BK_val = month_data_BK.sel(member=members)[true_idx.sel(member=members)]

        standardized_anomalies = (temperature_values - climatology) / std

        ax.scatter(
            temperature_values.time,
            standardized_anomalies,
            marker='o',
            c=BK_val,
            cmap=cmap,
            norm=norm
        )

    ax.set(
        ylabel="Standard deviation [σ]",
        title=months[m - 1].capitalize(),
        xlim=(11000, 50000)
    )
    ax.grid(True)

plt.suptitle("All CWMs Compared To Climatology σ (1971–2000 climatology)", fontsize=16)

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, 100, 11))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, 100, 11)])

plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.show()

fig.savefig("CWMsSigmaClim_DJF.png", bbox_inches="tight", dpi=300)

##### 3 coldest

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 14))
axes = axes.flatten()  
norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis
sm = ScalarMappable(norm=norm, cmap=cmap)

symbols_test = ['x','o','.']
label_test = ['1st','2nd','3rd']
for i in range(1, 13):  
    ax = axes[i - 1]  
    for j in range(n):
        BK_val = min_BK_da.sel(month=i,rank = j)

        coldest_vals = min_vals_da.sel(month=i, rank = j)
        climatology = climate_normal.sel(month=i)
        
        std = []
        for members in members_list:
    
            std.append(combined_member_std.sel(month = i, member = members))
    
        std_dev = std
        
        standardized_anomalies = (coldest_vals - climatology ) / std_dev
    
        ax.scatter(coldest_vals['time'], standardized_anomalies, marker = symbols_test[j] ,c=BK_val, cmap=cmap, norm=norm, label = label_test[j])
    ax.set(ylabel="Standard deviation [σ]",title = months[i - 1].capitalize(),xlim=(11000, 50000))
    ax.grid(True)
    ax.legend(loc='lower right')
plt.suptitle("Standardized Coldest Month Anomalies Internal σ (1971–2000 climatology)", fontsize=16)

fig.subplots_adjust(right=0.88)  
cbar_ax = fig.add_axes([1.0, 0.15, 0.02, 0.7])  
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, 100, 11))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, 100, 11)])
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

fig.savefig("ColdestInternal.png", bbox_inches="tight", dpi=300)

In [ ]:
winter_months = [12, 1, 2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis
sm = ScalarMappable(norm=norm, cmap=cmap)

symbols_test = ['x','o','.']
label_test = ['1st','2nd','3rd']

for idx, month in enumerate(winter_months):
    ax = axes[idx]
    
    for j in range(n):
        BK_val = min_BK_da.sel(month=month, rank=j)
        coldest_vals = min_vals_da.sel(month=month, rank=j)
        climatology = climate_normal.sel(month=month)

        std = []
        for member in members_list:
            std.append(combined_member_std.sel(month=month, member=member))
        std_dev = std
        
        standardized_anomalies = (coldest_vals - climatology) / std_dev

        ax.scatter(
            coldest_vals['time'], standardized_anomalies,
            marker=symbols_test[j], c=BK_val, cmap=cmap, norm=norm,
            label=label_test[j]
        )

    ax.set(
        ylabel="Standard deviation [σ]",
        title=months[month - 1].capitalize(),
        xlim=(11000, 50000)
    )
    ax.grid(True)
    ax.legend(loc='lower right')

plt.suptitle("Standardized Coldest Month Anomalies Internal σ (1971–2000 climatology)", fontsize=16)

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, 100, 11))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, 100, 11)])

plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.show()
fig.savefig("ColdestInternal_DJF.png", bbox_inches="tight", dpi=300)


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 14))
axes = axes.flatten()  
norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis

for i in range(1, 13):  
    ax = axes[i - 1]  
    for j in range(n):
        BK_val = min_BK_da.sel(month=i,rank = j)
        coldest_vals = min_vals_da.sel(month=i,rank = j)
        climatology = climate_normal.sel(month=i)
        std_dev = climate_std_within_members.sel(month=i)
    
        standardized_anomalies = (coldest_vals - climatology ) / std_dev
    
        max_time_idx = coldest_vals['time'].argmax()
        max_time_idx = coldest_vals.argmin()

        ax.scatter(coldest_vals['time'], standardized_anomalies, marker = symbols_test[j] ,c=BK_val, cmap=cmap, norm=norm, label = label_test[j])

    ax.set(ylabel="Standard deviation [σ]",title = months[i - 1].capitalize(),xlim=(11000, 50000))
    ax.legend(loc='lower right')

    ax.grid(True)

sm = ScalarMappable(norm=norm, cmap=cmap)
fig.subplots_adjust(right=0.88)  #colorbar
cbar_ax = fig.add_axes([1.0, 0.15, 0.02, 0.7])  
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, norm.vmax, 10))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, norm.vmax, 10)])

plt.suptitle("Standardized Coldest Month Anomalies External σ (1971-2000 climatology)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
fig.savefig("ColdestExternal.png", bbox_inches="tight", dpi=300)

In [ ]:
winter_months = [12, 1, 2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()
norm = Normalize(vmin=BK_SIC.min(), vmax=BK_SIC.max())
cmap = plt.cm.viridis

for plot_idx, month_i in enumerate(winter_months):
    ax = axes[plot_idx]

    for j in range(n):
        BK_val = min_BK_da.sel(month=month_i, rank=j)
        coldest_vals = min_vals_da.sel(month=month_i, rank=j)
        climatology = climate_normal.sel(month=month_i)
        std_dev = climate_std_within_members.sel(month=month_i)

        standardized_anomalies = (coldest_vals - climatology) / std_dev

        ax.scatter(
            coldest_vals['time'],
            standardized_anomalies,
            marker=symbols_test[j],
            c=BK_val,
            cmap=cmap,
            norm=norm,
            label=label_test[j]
        )

    ax.set(
        ylabel="Standard deviation [σ]",
        title=months[month_i - 1].capitalize(),
        xlim=(11000, 50000)
    )
    ax.legend(loc='lower right')
    ax.grid(True)

sm = ScalarMappable(norm=norm, cmap=cmap)
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([1.0, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("BK SIC [%]")
cbar.set_ticks(np.linspace(norm.vmin, norm.vmax, 10))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(norm.vmin, norm.vmax, 10)])

plt.suptitle("Standardized Coldest Month Anomalies External σ for DJF (1971–2000)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
fig.savefig("ColdestExternal_DJF.png", bbox_inches="tight", dpi=300)


In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(10, 14))
axes = axes.flatten()  
norm = Normalize(vmin=BK_SIC_fut['time'].min(), vmax=BK_SIC_fut['time'].max())
cmap = plt.cm.plasma
sm = ScalarMappable(norm=norm, cmap=cmap)

for i in range(1, 13):  
    ax = axes[i - 1]  
    for j in range(n):
        BK_val = min_BK_da.sel(month=i,rank = j)
        NP_val = min_NPL_da.sel(month=i,rank = j)

        
        ax.scatter(BK_val, NP_val, c=BK_val['time'],marker = symbols_test[j], cmap=cmap, norm=norm)

    ax.set(ylabel="NP SIC [%]",xlabel = "BK SIC [%]",title = months[i - 1].capitalize())
    ax.grid(True)
plt.suptitle("Sea ice concentration for 3 Coldest Month", fontsize=16)

fig.subplots_adjust(right=0.88)  #colorbar
cbar_ax = fig.add_axes([1.0, 0.15, 0.02, 0.7])  
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Time")
cbar.set_ticks(np.linspace(norm.vmin, norm.vmax, 10))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(2006, 2100, 10)])
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
fig.savefig("ColdestSeaIce.png", bbox_inches="tight", dpi=300)

In [ ]:
# Only M12, M1, M2
winter_months = [12, 1, 2]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes = axes.flatten()

norm = Normalize(vmin=BK_SIC_fut['time'].min(), vmax=BK_SIC_fut['time'].max())
cmap = plt.cm.plasma
sm = ScalarMappable(norm=norm, cmap=cmap)

for idx, month in enumerate(winter_months):
    ax = axes[idx]
    
    for j in range(n):
        BK_val = min_BK_da.sel(month=month, rank=j)
        NP_val = min_NPL_da.sel(month=month, rank=j)

        ax.scatter(
            BK_val, NP_val,
            c=BK_val['time'],
            marker=symbols_test[j],
            cmap=cmap,
            norm=norm
        )

    ax.set(
        ylabel="NP SIC [%]",
        xlabel="BK SIC [%]",
        title=months[month - 1].capitalize()
    )
    ax.grid(True)

plt.suptitle("Sea ice concentration for Coldest Month Anomalies", fontsize=16)

# Colorbar
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Time")
cbar.set_ticks(np.linspace(norm.vmin, norm.vmax, 10))
cbar.set_ticklabels([f"{int(y)}" for y in np.linspace(2006, 2100, 10)])

plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.show()
fig.savefig("ColdestSeaIce_DJF.png", bbox_inches="tight", dpi=300)

#### 2x2 plots

In [ ]:
base_path = ["Data/SR/r106","Data/SR/r139","Data/SR/r117"]
model_names = ["r106","r139","r117"]
time_stamp = ["2079-12-16T12:00:00","2077-01-16T12:00:00","2069-02-15"]

for i in range(3):
    file_path1 = os.path.join(base_path[i], "ts", f"ts_t_{model_names[i]}.nc")
    file_path2 = os.path.join(base_path[i], "sic", f"sic_t_{model_names[i]}.nc")
    file_path3 = os.path.join(base_path[i], f"hfls_t_{model_names[i]}.nc")
    file_path4 = os.path.join(base_path[i], f"zg_f_{model_names[i]}.nc")

    ds1 = xr.open_dataset(file_path1)
    ds2 = xr.open_dataset(file_path2)
    ds3 = xr.open_dataset(file_path3)
    ds4 = xr.open_dataset(file_path4)
    
    lon2d = ds1['lon']
    lat2d = ds1['lat']

    lon2d_ocean = ds2['longitude']
    lat2d_ocean = ds2['latitude']
    
    base_date = pd.to_datetime(time_stamp[i])
    
    month_offsets = [-2, -1, 0, 1]
    
    print("Running for model: ", model_names[i])
    for offset in month_offsets:
        
        current_date = base_date + pd.DateOffset(months=offset)
        current_time_read = current_date.strftime("%Y-%m")
        
        siconc_future = ds2['siconc'].sel(time=current_date, method='nearest').squeeze()
        tas = ds1['tas'].sel(time=current_date, method='nearest').squeeze() - 273.15
        hfls = ds3['hfls'].sel(time=current_date, method='nearest').squeeze()
        zg = ds4['zg'].sel(time=current_date, method='nearest').squeeze()
        
        proj = ccrs.Orthographic(central_longitude=15, central_latitude=45)
        fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 10), subplot_kw={'projection': proj})
        
        
        ax1.set_global()
        ax1.coastlines(alpha = 0.5)
        ax1.set_facecolor('lightgray')
        ax1.set_title(f"Temp (°C) for {model_names[i]} at {current_time_read} ")
        
        mesh1 = ax1.pcolormesh(
            lon2d, lat2d, tas,
            transform=ccrs.PlateCarree(),
            cmap='RdBu_r',
            vmin=-30, vmax=30
        )
        
        tas_min = float(tas.min())
        tas_max = float(tas.max())
        levels_tas = np.arange(np.floor(tas_min / 5) * 5, np.ceil(tas_max / 5) * 5 + 5, 5)
    
        contours_tas = ax1.contour(
            lon2d, lat2d, tas,
            levels=levels_tas,
            colors='black',
            linewidths=0.8,
            transform=ccrs.PlateCarree(),
            alpha=0.5 
        )
        
        highlight_level_tas = [0]
        highlight_contour_tas = ax1.contour(
            lon2d, lat2d, tas,
            levels=highlight_level_tas,
            colors='black',          
            linewidths=1.2,        
            transform=ccrs.PlateCarree(),
            alpha=0.9              
        )
        
        ax1.clabel(contours_tas, inline=True, fontsize=8, fmt='%1.0f')
        fig.colorbar(mesh1, ax=ax1, orientation='vertical', label='Temperature (°C)', fraction=0.046, pad=0.04)
        
        
        ax2 = axs[0, 1]
        ax2.set_global()
        ax2.set_facecolor('lightgray')
        ax2.set_title(f"Sea Ice Concentraion (%) for {model_names[i]} at {current_time_read} ")
        
        mesh2 = ax2.pcolormesh(
            lon2d_ocean, lat2d_ocean, siconc_future,
            transform=ccrs.PlateCarree(),
            cmap='viridis'
        )
        
        fig.colorbar(mesh2, ax=ax2, orientation='vertical', label='Sea Ice (%)', fraction=0.046, pad=0.04)
        
        
        ax3 = axs[1, 0]
        ax3.set_global()
        ax3.set_facecolor('lightgray')
        ax3.set_title(f"Surface Upward Latent Heat Flux (W m$^{{-2}}$) for {model_names[i]} at {current_time_read} ")
        
        mesh3 = ax3.pcolormesh(
            lon2d, lat2d, hfls,
            transform=ccrs.PlateCarree(),
            cmap='viridis',
            vmin=0, vmax=550
        )
        fig.colorbar(mesh3, ax=ax3, orientation='vertical', label='Surface Upward Latent Heat Flux [W m-2]', fraction=0.046, pad=0.04)

        
        ax4 = axs[1, 1]
        ax4.set_global()
        ax4.coastlines()
        ax4.set_title(f"Geopotential height 500 hPa {model_names[i]} at {current_time_read}")

        mesh4 = ax4.pcolormesh(
            lon2d, lat2d, zg,
            transform=ccrs.PlateCarree(),
            cmap='viridis',
            vmin=5150, vmax=5900
        )
        
        zg_min = float(zg.min())
        zg_max = float(zg.max())
        levels = np.arange(np.floor(zg_min / 40) * 40, np.ceil(zg_max / 40) * 40 + 40, 40)

        contours = ax4.contour(
            lon2d, lat2d, zg,
            levels=levels,
            colors='black',
            linewidths=0.8,
            transform=ccrs.PlateCarree(),
            alpha=0.7 
        )

        highlight_level = [5520]
        highlight_contour = ax4.contour(
            lon2d, lat2d, zg,
            levels=highlight_level,
            colors='black',          
            linewidths=1.2,        
            transform=ccrs.PlateCarree(),
            alpha=0.9              
        )
        
        ax4.clabel(contours, inline=True, fontsize=8, fmt='%1.0f')

        fig.colorbar(mesh4, ax=ax4, orientation='vertical', label='Geopotential height', fraction=0.046, pad=0.04)

        plt.tight_layout()
        
        fig.savefig(f"climate_{model_names[i]}_{current_time_read}.png", bbox_inches="tight", dpi=125)
        plt.show()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import os
import cartopy.crs as ccrs
from dask.diagnostics import ProgressBar

# 1. Define the split members
sr_ids = [int(d[1:]) for d in os.listdir("Data/SR") if d.startswith("r")]
lr_ids = [int(d[1:]) for d in os.listdir("Data/LR") if d.startswith("r")]

variables = ["ts", "sic", "hfls", "zg"]
base_dir = "Data"

# 2. Preprocessing: Now with DJF filtering!
def preprocess_to_map(ds):
    # Slice the time period
    ds_sliced = ds.sel(time=slice("1970", "2000"))
    # Filter for only Winter months (December, January, February)
    ds_djf = ds_sliced.sel(time=ds_sliced['time.season'] == 'DJF')
    # Average the remaining DJF months into a single map
    return ds_djf.mean(dim="time")

# Set up the Cartopy projection and Figure
proj = ccrs.Orthographic(central_longitude=15, central_latitude=45)
fig, axes = plt.subplots(2, 2, figsize=(16, 12), subplot_kw={'projection': proj})
axes = axes.flatten()

# 3. Process Variables
for i, var in enumerate(variables):
    file_list = []
    
    # Check SR members
    for m in sr_ids:
        m_str = f"r{m}"
        subdir = var if var in ["ts", "sic"] else ""
        path = os.path.join(base_dir, "SR", m_str, subdir, f"{var}_t_{m_str}.nc")
        if os.path.exists(path): file_list.append(path)

    # Check LR members
    for m in lr_ids:
        m_str = f"r{m}"
        subdir = var if var in ["ts", "sic"] else ""
        path = os.path.join(base_dir, "LR", m_str, subdir, f"{var}_t_{m_str}.nc")
        if os.path.exists(path): file_list.append(path)

    print(f"\nFound {len(file_list)} files for {var}.")
    if not file_list:
        continue

    # 4. Open with Dask
    ds_ens = xr.open_mfdataset(
        file_list, 
        concat_dim="ensemble", 
        combine="nested",
        preprocess=preprocess_to_map,
        chunks='auto', 
    )

    # Map the internal NetCDF variables
    if var == "ts":
        internal_var = "tas"
    elif var == "sic":
        internal_var = "siconc"
    else:
        internal_var = var

    # 5. Compute the Ensemble Mean with Progress Bar
    print(f"Calculating DJF ensemble mean for {internal_var}...")
    with ProgressBar():
        ens_mean = ds_ens[internal_var].mean(dim="ensemble").compute()

    # Temperature adjustment (Kelvin to Celsius)
    if internal_var == "tas":
        ens_mean = ens_mean - 273.15

    # 6. Extract the correct 2D coordinates for plotting
    if var == "sic":
        lon2d = ds_ens['longitude'].values
        lat2d = ds_ens['latitude'].values
    else:
        lon2d = ds_ens['lon'].values
        lat2d = ds_ens['lat'].values

    # 7. Plotting on the Map
    ax = axes[i]
    ax.set_global()
    ax.set_facecolor('lightgray')
    ax.coastlines()
    
    cmap = 'RdBu_r' if internal_var == "tas" else 'viridis'

    # --- THE FIX: We add .squeeze() to drop the length-1 'plev' dimension from zg ---
    plot_data = ens_mean.squeeze().values

    mesh = ax.pcolormesh(
        lon2d, lat2d, plot_data,
        transform=ccrs.PlateCarree(),
        cmap=cmap
    )
    
    fig.colorbar(mesh, ax=ax, orientation='vertical', label=internal_var, fraction=0.046, pad=0.04)
    ax.set_title(f"EC-Earth3 {internal_var} DJF Mean: 1970-2000 (N={len(file_list)})")

plt.tight_layout()
plt.show()

#### 2x2 Differences

In [ ]:
from dask.diagnostics import ProgressBar

print("--- Calculating 57-Member Historical Monthly Means (O, N, D, J, F, M) ---")

sr_ids = [int(d[1:]) for d in os.listdir("Data/SR") if d.startswith("r")]
lr_ids = [int(d[1:]) for d in os.listdir("Data/LR") if d.startswith("r")]
variables = ["ts", "sic", "hfls", "zg"]
base_dir = "Data"

def preprocess_to_map(ds):
    ds_sliced = ds.sel(time=slice("1971", "2000"))
    ds_ondjf = ds_sliced.sel(time=ds_sliced['time.month'].isin([10, 11, 12, 1, 2, 3]))
    return ds_ondjf.groupby('time.month').mean(dim="time").squeeze()

print("\n--- Plotting Anomalies ---")

base_path = ["Data/SR/r106","Data/SR/r139","Data/SR/r117"]
model_names = ["r106","r139","r117"]
time_stamp = ["2079-12-16T12:00:00","2077-01-16T12:00:00","2069-02-15"]
robust_hist_means = {}

for var in variables:
    file_list = []
    
    for m in sr_ids:
        path = os.path.join(base_dir, "SR", f"r{m}", var if var in ["ts", "sic"] else "", f"{var}_t_r{m}.nc")
        if os.path.exists(path): file_list.append(path)
    for m in lr_ids:
        path = os.path.join(base_dir, "LR", f"r{m}", var if var in ["ts", "sic"] else "", f"{var}_t_r{m}.nc")
        if os.path.exists(path): file_list.append(path)

    if not file_list: continue

    ds_ens = xr.open_mfdataset(
        file_list, concat_dim="ensemble", combine="nested",
        preprocess=preprocess_to_map, chunks='auto'
    )

    internal_var = "tas" if var == "ts" else "siconc" if var == "sic" else var
    
    print(f"Computing master mean for {internal_var}...")
    with ProgressBar():
        robust_hist_means[internal_var] = ds_ens[internal_var].mean(dim="ensemble").compute()


for i in range(3):
    file_path1 = os.path.join(base_path[i], "ts", f"ts_t_{model_names[i]}.nc")
    file_path2 = os.path.join(base_path[i], "sic", f"sic_t_{model_names[i]}.nc")
    file_path3 = os.path.join(base_path[i], f"hfls_t_{model_names[i]}.nc")
    file_path4 = os.path.join(base_path[i], f"zg_f_{model_names[i]}.nc")

    ds1 = xr.open_dataset(file_path1)
    ds2 = xr.open_dataset(file_path2)
    ds3 = xr.open_dataset(file_path3)
    ds4 = xr.open_dataset(file_path4)
    
    lon2d, lat2d = ds1['lon'].values, ds1['lat'].values
    lon2d_ocean, lat2d_ocean = ds2['longitude'].values, ds2['latitude'].values
    
    base_date = pd.to_datetime(time_stamp[i])
    month_offsets = [-2, -1, 0, 1]

    print("Running for model: ", model_names[i])
    for offset in month_offsets:
        current_date = base_date + pd.DateOffset(months=offset)
        current_time_read = current_date.strftime("%Y-%m")
        target_month = current_date.month

        tas_fut = ds1['tas'].sel(time=current_time_read).squeeze()
        sic_fut = ds2['siconc'].sel(time=current_time_read).squeeze()
        hfls_fut = ds3['hfls'].sel(time=current_time_read).squeeze()
        zg_fut = ds4['zg'].sel(time=current_time_read).squeeze()

        
        z_eu = eu_zscores.sel(member=model_names[i], time=current_time_read).squeeze().item()
        z_sic = bk_sic_zscores.sel(member=model_names[i], time=current_time_read).squeeze().item()
        z_sic_np = np_sic_zscores.sel(member=model_names[i], time=current_time_read).squeeze().item()
        z_hfls = bk_hfls_zscores.sel(member=model_names[i], time=current_time_read).squeeze().item()
        z_zg_eu = eu_zg_zscores.sel(member=model_names[i], time=current_time_read).squeeze().item()
        z_zg_bk = bk_zg_zscores.sel(member=model_names[i], time=current_time_read).squeeze().item()
        z_zg_np = np_zg_zscores.sel(member=model_names[i], time=current_time_read).squeeze().item()
        
        tas_anom = (tas_fut - robust_hist_means['tas'].sel(month=target_month)).values
        sic_anom = (sic_fut - robust_hist_means['siconc'].sel(month=target_month)).values
        hfls_anom = (hfls_fut - robust_hist_means['hfls'].sel(month=target_month)).values
        zg_anom = (zg_fut - robust_hist_means['zg'].sel(month=target_month)).values
        
        proj = ccrs.Orthographic(central_longitude=15, central_latitude=45)
        fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(14, 12), subplot_kw={'projection': proj})
        
        ax1 = axs[0, 0]
        ax1.set_global()
        ax1.set_facecolor('lightgray')
        ax1.coastlines(alpha = 0.5)
        ax1.set_title(f"Temp Anomaly (°C) for {model_names[i]} at {current_time_read}\nExtremity EU: {z_eu:.2f}", fontweight='bold')
        
        mesh1 = ax1.pcolormesh(
            lon2d, lat2d, tas_anom,
            transform=ccrs.PlateCarree(),
            cmap='RdBu_r', vmin=-15, vmax=15
        )
        
        tas_min = float(tas_anom.min())
        tas_max = float(tas_anom.max())
        levels_tas = np.arange(np.floor(tas_min / 5) * 5, np.ceil(tas_max / 5) * 5 + 5, 5)

        contours_tas = ax1.contour(
            lon2d, lat2d, tas_anom,
            levels=levels_tas,
            colors='black',
            linewidths=0.8,
            transform=ccrs.PlateCarree(),
            alpha=0.5 
        )

        highlight_level_tas = [0]
        highlight_contour_tas = ax1.contour(
            lon2d, lat2d, tas_anom,
            levels=highlight_level_tas,
            colors='black',          
            linewidths=1.2,        
            transform=ccrs.PlateCarree(),
            alpha=0.9              
        )

        ax1.clabel(contours_tas, inline=True, fontsize=8, fmt='%1.0f')
        
        fig.colorbar(mesh1, ax=ax1, orientation='vertical', label='Δ Temperature (°C)', fraction=0.046, pad=0.04)
        
        ax2 = axs[0, 1]
        ax2.set_global()
        ax2.set_facecolor('lightgray')
        ax2.coastlines()
        ax2.set_title(f"Sea Ice Anomaly (%) for {model_names[i]} at {current_time_read}\nExtremity BK: {z_sic:.2f} | NP: {z_sic_np:.2f}", fontweight='bold')
        
        mesh2 = ax2.pcolormesh(
            lon2d_ocean, lat2d_ocean, sic_anom,
            transform=ccrs.PlateCarree(),
            cmap='RdBu', vmin=-100, vmax=100
        )
        fig.colorbar(mesh2, ax=ax2, orientation='vertical', label='Δ Sea Ice (%)', fraction=0.046, pad=0.04)
        
        ax3 = axs[1, 0]
        ax3.set_global()
        ax3.set_facecolor('lightgray')
        ax3.coastlines()
        ax3.set_title(f"Latent Heat Flux Anomaly (W m$^{{-2}}$) for {model_names[i]} at {current_time_read}\nExtremity BK: {z_hfls:.2f}", fontweight='bold')
        
        mesh3 = ax3.pcolormesh(
            lon2d, lat2d, hfls_anom,
            transform=ccrs.PlateCarree(),
            cmap='RdBu_r', vmin=-200, vmax=200
        )
        fig.colorbar(mesh3, ax=ax3, orientation='vertical', label='Δ LHF [W m-2]', fraction=0.046, pad=0.04)

        ax4 = axs[1, 1]
        ax4.set_global()
        ax4.set_facecolor('lightgray')
        ax4.coastlines()
        ax4.set_title(f"Z500 Anomaly for {model_names[i]} at {current_time_read}\nExtremity EU: {z_zg_eu:.2f} | BK: {z_zg_bk:.2f} | NP: {z_zg_np:.2f}", fontweight='bold')

        mesh4 = ax4.pcolormesh(
            lon2d, lat2d, zg_anom,
            transform=ccrs.PlateCarree(),
            cmap='RdBu_r', vmin=-350, vmax=350
        )
        
        zg_min = float(zg_anom.min())
        zg_max = float(zg_anom.max())
        levels_zg = np.arange(np.floor(zg_min / 40) * 40, np.ceil(zg_max / 40) * 40 + 40, 40)

        contours_zg = ax4.contour(
            lon2d, lat2d, zg_anom,
            levels=levels_zg,
            colors='black',
            linewidths=0.8,
            transform=ccrs.PlateCarree(),
            alpha=0.7 
        )

        highlight_level = [0]
        highlight_contour = ax4.contour(
            lon2d, lat2d, zg_anom,
            levels=highlight_level,
            colors='black',          
            linewidths=1.2,        
            transform=ccrs.PlateCarree(),
            alpha=0.9              
        )

        ax4.clabel(contours_zg, inline=True, fontsize=8, fmt='%1.0f')
        
        fig.colorbar(mesh4, ax=ax4, orientation='vertical', label='Δ Geopotential height', fraction=0.046, pad=0.04)
        plt.tight_layout()
        

        fig.savefig(f"climate_difference_{model_names[i]}_{current_time_read}.png", bbox_inches="tight", dpi=125)
        plt.show()

#### Seasonal cycle

In [ ]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
          'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

month_labels = [m.capitalize() for m in months]

monthly_mean_vals = [] 
monthly_mean_vals_time = []
monthly_mean_vals_diff_t = []
monthly_mean_vals_i_t = [] 

for i in range(1,13):
    monthly_mean_vals.append(europe_temp.sel(time=slice("1971-01", "2000-12")).groupby("time.month")[i].mean(dim = 'member').mean().values)
    monthly_mean_vals_time.append(i)

fig, ax = plt.subplots(nrows= 1,ncols = 2,figsize=(16,8))


for members in members_list:
    monthly_mean_vals_run = []
    for i in range(1,13):
        monthly_mean_vals_run.append(
            europe_temp.sel(time=slice("1971-01", "2000-12"))
            .groupby("time.month")[i]
            .sel(member=members)
            .mean()
            .values)
        
    diff = np.array(monthly_mean_vals_run) - np.array(monthly_mean_vals)
    monthly_mean_vals_diff_t.append(diff)
    monthly_mean_vals_i_t.append(monthly_mean_vals_run)
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_run, c='gray', alpha=0.5)


monthly_mean_vals_diff = xr.DataArray(
    np.array(monthly_mean_vals_diff_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="diff",
)

monthly_mean_vals_i = xr.DataArray(
    np.array(monthly_mean_vals_i_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="val",
)

ax[1].boxplot(monthly_mean_vals_diff, positions = monthly_mean_vals_time);


monthly_mean_vals_diff_sum = monthly_mean_vals_diff.sum(dim='month').values
idx_large = np.argsort(monthly_mean_vals_diff_sum)[-5:]
idx_small = np.argsort(S)[:5]

winter = monthly_mean_vals_diff.sel(month=[12,1,2,3]).sum(dim="month")
summer = monthly_mean_vals_diff.sel(month=[6,7,8,9]).sum(dim="month")

mask = (winter > 0) & (summer < 0)

for i in range(5):
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i]], alpha = 0.5, color='red')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i]],alpha = 0.5, color='red')
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i]], alpha = 0.5, color='blue')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i]],alpha = 0.5, color='blue')
        
   
    if i + 1 == 1: 
        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals, '-.',marker = 'o', color='k', label='Climate normal mean')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')
        

for i in range(2):
    ax[i].set_xticks(ticks=range(1, 13), labels= month_labels, fontsize=14)
    ax[i].set_xlabel("Month")
    ax[i].grid()
    ax[i].legend()

plt.suptitle("Seasonal cycle (1971–2000 baseline)", fontsize=16)

ax[0].set_ylabel("Temp (°C)", fontsize=18)
ax[1].set_ylabel("Temp difference (°C)", fontsize=18)
plt.savefig("Seasonal_Cycle.png", bbox_inches="tight", dpi=300)

In [ ]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
          'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

month_labels = [m.capitalize() for m in months]

monthly_mean_vals = [] 
monthly_mean_vals_time = []
monthly_mean_vals_diff_t = []
monthly_mean_vals_i_t = [] 

for i in range(1,13):
    monthly_mean_vals.append(europe_temp.sel(time=slice("1990-01", "2014-12")).groupby("time.month")[i].mean(dim = 'member').mean().values)
    monthly_mean_vals_time.append(i)

fig, ax = plt.subplots(nrows= 1,ncols = 2,figsize=(16,8))


for members in members_list:
    monthly_mean_vals_run = []
    for i in range(1,13):
        monthly_mean_vals_run.append(
            europe_temp.sel(time=slice("1980-01", "2014-12"))
            .groupby("time.month")[i]
            .sel(member=members)
            .mean()
            .values)
        
    diff = np.array(monthly_mean_vals_run) - np.array(monthly_mean_vals)
    monthly_mean_vals_diff_t.append(diff)
    monthly_mean_vals_i_t.append(monthly_mean_vals_run)
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_run, c='gray', alpha=0.5)


monthly_mean_vals_diff = xr.DataArray(
    np.array(monthly_mean_vals_diff_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="diff",
)

monthly_mean_vals_i = xr.DataArray(
    np.array(monthly_mean_vals_i_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="val",
)


monthly_mean_vals_diff_sum = monthly_mean_vals_diff.sum(dim='month').values
idx_large = np.argsort(monthly_mean_vals_diff_sum)[-5:]
idx_small = np.argsort(monthly_mean_vals_diff_sum)[:5]

winter = monthly_mean_vals_diff.sel(month=[12,1,2,3]).sum(dim="month")
summer = monthly_mean_vals_diff.sel(month=[6,7,8,9]).sum(dim="month")

mask = (winter > 0) & (summer < 0)
members_dampened_idx = np.argwhere(mask.values).flatten()


for i in range(len(members_dampened_idx)):
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i]], alpha = 0.5, color='red')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i]],alpha = 0.5, color='red')
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i]], alpha = 0.5, color='blue')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i]],alpha = 0.5, color='blue')
        
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[members_dampened_idx[i]], alpha = 0.5, color='purple')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[members_dampened_idx[i]],alpha = 0.5, color='purple')

    if i + 1 == len(members_dampened_idx):
        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals, '-.',marker = 'o', color='k', label='Climate normal mean')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')
        
ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[members_dampened_idx[0]], alpha = 0.5, color='purple',label = 'Dampened')
ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[members_dampened_idx[0]],alpha = 0.5, color='purple',label = 'Dampened')

ax[1].axhline(0 ,c='k',  label = 'Climate normal mean')


for i in range(2):
    ax[i].set_xticks(ticks=range(1, 13), labels= month_labels, fontsize=14)
    ax[i].set_xlabel("Month")
    ax[i].grid()
    ax[i].legend()

# plt.suptitle("Seasonal cycle (1990–2014 baseline)", fontsize=16)

ax[0].set_ylabel("Temp (°C)", fontsize=18)
ax[1].set_ylabel("Temp difference (°C)", fontsize=18)

In [ ]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
          'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

month_labels = [m.capitalize() for m in months]

monthly_mean_vals = [] 
monthly_mean_vals_time = []
monthly_mean_vals_diff_t = []
monthly_mean_vals_i_t = []

for i in range(1,13):
    monthly_mean_vals.append(europe_temp.sel(time=slice("2006-01", "2050-12")).groupby("time.month")[i].mean(dim = 'member').mean().values)
    monthly_mean_vals_time.append(i)

fig, ax = plt.subplots(nrows= 1,ncols = 2,figsize=(16,8))


for members in members_list:
    monthly_mean_vals_run = []
    for i in range(1,13):
        monthly_mean_vals_run.append(
            europe_temp.sel(time=slice("2015-01", "2049-12"))
            .groupby("time.month")[i]
            .sel(member=members)
            .mean()
            .values)
        
    diff = np.array(monthly_mean_vals_run) - np.array(monthly_mean_vals)
    monthly_mean_vals_diff_t.append(diff)
    monthly_mean_vals_i_t.append(monthly_mean_vals_run)
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_run, c='gray', alpha=0.5)
    ax[1].plot(monthly_mean_vals_time, diff, c='gray', alpha=0.35)



monthly_mean_vals_diff = xr.DataArray(
    np.array(monthly_mean_vals_diff_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="diff",
)

monthly_mean_vals_i = xr.DataArray(
    np.array(monthly_mean_vals_i_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="val",
)

for i in range(len(members_dampened_idx)):
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i]], alpha = 0.5, color='red')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i]],alpha = 0.5, color='red')
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i]], alpha = 0.5, color='blue')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i]],alpha = 0.5, color='blue')
        
    if i + 1 == len(members_dampened_idx): 
        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals, '-.',marker = 'o', color='k', label='Climate normal mean')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')
        

ax[1].axhline(0 ,c='k',  label = 'Climate normal mean')


for i in range(2):
    ax[i].set_xticks(ticks=range(1, 13), labels= month_labels, fontsize=14)
    ax[i].set_xlabel("Month")
    ax[i].grid()
    ax[i].legend()

plt.suptitle("Seasonal cycle (2006-2050 baseline)", fontsize=16)

ax[0].set_ylabel("Temp (°C)", fontsize=18)
ax[1].set_ylabel("Temp difference (°C)", fontsize=18)
plt.savefig("SeasonalCycleNearFut.png", bbox_inches="tight", dpi=300)

In [ ]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
          'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

month_labels = [m.capitalize() for m in months]

monthly_mean_vals = [] 
monthly_mean_vals_time = []
monthly_mean_vals_diff_t = []
monthly_mean_vals_i_t = []

for i in range(1,13):
    monthly_mean_vals.append(europe_temp.sel(time=slice("2051-01", "2100-12")).groupby("time.month")[i].mean(dim = 'member').mean().values)
    monthly_mean_vals_time.append(i)

fig, ax = plt.subplots(nrows= 1,ncols = 2,figsize=(16,8))


for members in members_list:
    monthly_mean_vals_run = []
    for i in range(1,13):
        monthly_mean_vals_run.append(
            europe_temp.sel(time=slice("2050-01", "2100-12"))
            .groupby("time.month")[i]
            .sel(member=members)
            .mean()
            .values)
        
    diff = np.array(monthly_mean_vals_run) - np.array(monthly_mean_vals)
    monthly_mean_vals_diff_t.append(diff)
    monthly_mean_vals_i_t.append(monthly_mean_vals_run)
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_run, c='gray', alpha=0.5)
    ax[1].plot(monthly_mean_vals_time, diff, c='gray', alpha=0.35)

monthly_mean_vals_diff = xr.DataArray(
    np.array(monthly_mean_vals_diff_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="diff",
)

monthly_mean_vals_i = xr.DataArray(
    np.array(monthly_mean_vals_i_t),
    dims=["member", "month"],
    coords={
        "member": members_list,
        "month": np.arange(1, 13),
    },
    name="val",
)




for i in range(len(members_dampened_idx)):
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i]], alpha = 0.5, color='red')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i]],alpha = 0.5, color='red')
    
    ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i]], alpha = 0.5, color='blue')
    ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i]],alpha = 0.5, color='blue')
        
    if i + 1 == len(members_dampened_idx): 
        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals, '-.',marker = 'o', color='k', label='Climate normal mean')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_large[i+1]], alpha = 0.5, color='red', label='Warm')

        ax[0].plot(monthly_mean_vals_time, monthly_mean_vals_i[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')
        ax[1].plot(monthly_mean_vals_time, monthly_mean_vals_diff[idx_small[i+1]], alpha = 0.5, color='blue', label='Cold')

ax[1].axhline(0 ,c='k',  label = 'Climate normal mean')


for i in range(2):
    ax[i].set_xticks(ticks=range(1, 13), labels= month_labels, fontsize=14)
    ax[i].set_xlabel("Month")
    ax[i].grid()
    ax[i].legend()

plt.suptitle("Seasonal cycle (2051-2100 baseline)", fontsize=16)

ax[0].set_ylabel("Temp (°C)", fontsize=18)
ax[1].set_ylabel("Temp difference (°C)", fontsize=18)
plt.savefig("SeasonalCycleFarFut.png", bbox_inches="tight", dpi=300)

#### Recreating table 1

##### Paper Climatological

In [ ]:
europe_temp_clim_pap = europe_temp.sel(time=slice("1971-01", "2000-12"))

combined_member_mean_pap = europe_temp_clim_pap.groupby("time.month").mean(dim="time")

climate_normal_pap = combined_member_mean_pap.mean(dim="member")  

periods = [
    ("Climatological 1971-2000", "1971-01", "2000-12"),
    ("Historical 1970*-2005", "1970-01", "2005-12"),
    ("Future 1 2006-2050", "2006-01", "2050-12"),
    ("Future 2 2051-2100", "2051-01", "2100-12")
]

table = PrettyTable()
table.field_names = ["Member"] + [p[0] for p in periods]

clim_start, clim_end = periods[0][1], periods[0][2]
clim_counts = {}
for member in members_list:
    temp_clim = europe_temp.sel(member=member).sel(time=slice(clim_start, clim_end))
    true_count = 0
    for month in [12,1,2]:
        mask = temp_clim.groupby("time.month")[month] < climate_normal_pap.sel(month=month)
        true_count += int(np.sum(mask.values))
    clim_counts[member] = true_count

all_counts = {label: [] for label, _, _ in periods}
all_ratios = {label: [] for label, _, _ in periods}

for member in members_list:
    row = [str(member)]
    baseline = clim_counts[member]
    denom = 0
    for label, start, end in periods:
        temp = europe_temp.sel(member=member).sel(time=slice(start, end))
        true_count = 0
        for month in [12,1,2]:
            mask = temp.groupby("time.month")[month] < climate_normal_pap.sel(month=month)
            true_count += int(np.sum(mask.values))
        if baseline == 0:
            row.append(f"{true_count}(nan)")
        else:
            ratio = (true_count/(len(mask) * 3)) / (baseline/(90))
            row.append(f"{(true_count/(len(mask) * 3) * 100):.0f} ({ratio * 100:.0f})")
            all_counts[label].append(true_count)
            all_ratios[label].append(ratio)
    table.add_row(row)

mean_row = ["Mean"]
for label, _, _ in periods:
    if all_counts[label]:
        mean_count = np.mean(all_counts[label])
        mean_ratio = np.mean(all_ratios[label])
        mean_row.append(f"{mean_count:.0f}({mean_ratio:.2f})")
    else:
        mean_row.append("nan(nan)")

table.add_row(mean_row)
print(table)
print(table.get_latex_string())


In [ ]:
from prettytable import PrettyTable
periods = [
    ("Climatological 1990-2014", "1990-01", "2014-12"),
    ("Historical 1970-2014", "1970-01", "2014-12"),
    ("Future 1 2015-2050", "2015-01", "2050-12"),
    ("Future 2 2051-2100", "2051-01", "2100-12")
]

table = PrettyTable()
table.field_names = ["Member"] + [p[0] for p in periods]

clim_start, clim_end = periods[0][1], periods[0][2]
clim_counts = {}
for member in members_list:
    temp_clim = europe_temp.sel(member=member).sel(time=slice(clim_start, clim_end))
    true_count = 0
    for month in [12,1,2]:
        mask = temp_clim.groupby("time.month")[month] < climate_normal.sel(month=month)
        true_count += int(np.sum(mask.values))
    clim_counts[member] = true_count

all_counts = {label: [] for label, _, _ in periods}
all_ratios = {label: [] for label, _, _ in periods}

for member in members_list:
    row = [str(member)]
    baseline = clim_counts[member]
    for label, start, end in periods:
        temp = europe_temp.sel(member=member).sel(time=slice(start, end))
        true_count = 0
        for month in [12,1,2]:
            mask = temp.groupby("time.month")[month] < climate_normal.sel(month=month)
            true_count += int(np.sum(mask.values))
        if baseline == 0:
            row.append(f"{true_count}(nan)")
        else:
            ratio = true_count / baseline
            row.append(f"{true_count}({ratio:.2f})")
            all_counts[label].append(true_count)
            all_ratios[label].append(ratio)
    table.add_row(row)

mean_row = ["Mean"]
for label, _, _ in periods:
    if all_counts[label]:
        mean_count = np.mean(all_counts[label])
        mean_ratio = np.mean(all_ratios[label])
        mean_row.append(f"{mean_count:.0f}({mean_ratio:.2f})")
    else:
        mean_row.append("nan(nan)")

table.add_row(mean_row)

print(table)


#### Recreating table 2

##### Paper Climatological

In [ ]:
djf = SIC_BK_SAT.where(SIC_BK_SAT['time.month'].isin([12, 1, 2]), drop=True)

djf_SICSAT_d = SIC_BK_SAT.where(SIC_BK_SAT['time.month'].isin([12]), drop=True)
djf_SICSAT_j = SIC_BK_SAT.where(SIC_BK_SAT['time.month'].isin([1]), drop=True)
djf_SICSAT_f = SIC_BK_SAT.where(SIC_BK_SAT['time.month'].isin([2]), drop=True)

SICSAT_clim_d = djf_SICSAT_d.sel(time=slice("1971-01-01", "2000-12-31")).mean()
SICSAT_clim_j = djf_SICSAT_j.sel(time=slice("1971-01-01", "2000-12-31")).mean()
SICSAT_clim_f = djf_SICSAT_f.sel(time=slice("1971-01-01", "2000-12-31")).mean()

print(SICSAT_clim_d.values,SICSAT_clim_j.values,SICSAT_clim_f.values)
SICSAT_cli_d = (SICSAT_clim_d.values * 0.9 > djf_SICSAT_d.sel(time=slice("1971-01-01", "2000-12-31")))
SICSAT_cli_j = (SICSAT_clim_j.values * 0.9 > djf_SICSAT_j.sel(time=slice("1971-01-01", "2000-12-31")))
SICSAT_cli_f = (SICSAT_clim_f.values * 0.9 > djf_SICSAT_f.sel(time=slice("1971-01-01", "2000-12-31")))

SICSAT_hist_d = (SICSAT_clim_d.values * 0.9 > djf_SICSAT_d.sel(time=slice("1970-01-01", "2005-12-31")))
SICSAT_hist_j = (SICSAT_clim_j.values * 0.9 > djf_SICSAT_j.sel(time=slice("1970-01-01", "2005-12-31")))
SICSAT_hist_f = (SICSAT_clim_f.values * 0.9 > djf_SICSAT_f.sel(time=slice("1970-01-01", "2005-12-31")))

SICSAT_fut_d = (SICSAT_clim_d.values * 0.9 > djf_SICSAT_d.sel(time=slice("2006-01-01", "2025-12-31")))
SICSAT_fut_j = (SICSAT_clim_j.values * 0.9 > djf_SICSAT_j.sel(time=slice("2006-01-01", "2025-12-31")))
SICSAT_fut_f = (SICSAT_clim_f.values * 0.9 > djf_SICSAT_f.sel(time=slice("2006-01-01", "2025-12-31")))

prob_cli_SICSAT = (SICSAT_cli_d.values.sum() + SICSAT_cli_j.values.sum() + SICSAT_cli_f.values.sum()) / (len(SICSAT_hist_d) + len(SICSAT_hist_j) + len(SICSAT_hist_f))
prob_fut_SICSAT = (SICSAT_fut_d.values.sum() + SICSAT_fut_j.values.sum() + SICSAT_fut_f.values.sum()) / (len(SICSAT_hist_d) + len(SICSAT_hist_j) + len(SICSAT_hist_f))
prob_hist_SICSAT = (SICSAT_hist_d.values.sum() + SICSAT_hist_j.values.sum() + SICSAT_hist_f.values.sum()) / (len(SICSAT_hist_d) + len(SICSAT_hist_j) + len(SICSAT_hist_f))


In [ ]:
europe_temp_clim_pap = europe_temp.sel(time=slice("1971-01", "2000-12"))

combined_member_mean_pap = europe_temp_clim_pap.groupby("time.month").mean(dim="time")

climate_normal_pap = combined_member_mean_pap.mean(dim="member")  

periods = [
    ("Climatological 1971-2000", "1971-01", "2000-12"),
    ("Historical 1970*-2005", "1970-01", "2005-12"),
    ("Future  2006-2100", "2006-01", "2100-12")
]

table = PrettyTable()
table.field_names = ["Member"] + [p[0] for p in periods]

clim_start, clim_end = periods[0][1], periods[0][2]
clim_counts = {}

CWM_vals = {}
CWM_vals_np = {}
for member in members_list:
    temp_clim = europe_temp.sel(member=member).sel(time=slice(clim_start, clim_end))
    true_count = 0
    CWM_val = 0
    CWM_val_np = 0

    cwm_mask_all = []
    bk_event_mask_all = []
    np_event_mask_all = []

    for month in [12,1,2]:
        mask = temp_clim.groupby("time.month")[month] < climate_normal_pap.sel(month=month)
        BK_masked = BK_SIC.sel(member=member).sel(time=slice(clim_start, clim_end)).where(mask)
        NP_masked = NPL_SIC.sel(member=member).sel(time=slice(clim_start, clim_end)).where(mask)

        true_count += int(np.sum(mask.values))
        CWM_val += BK_masked.mean(skipna = True).values
        CWM_val_np += NP_masked.mean(skipna = True).values

    clim_counts[member] = true_count
    CWM_vals[member] = CWM_val/3 
    CWM_vals_np[member] = CWM_val_np/3 

bk_ratio_for_mean = {label: [] for label, _, _ in periods}
np_ratio_for_mean = {label: [] for label, _, _ in periods}

for member in members_list:
    row = [str(member)]
    baseline_bk = CWM_vals[member]
    baseline_bk = np.mean(list(CWM_vals.values()))

    baseline_np = CWM_vals_np[member]
    baseline_np = np.mean(list(CWM_vals_np.values()))

    
    for label, start, end in periods:
        temp = europe_temp.sel(member=member).sel(time=slice(start, end)).groupby("time.month")
        small_sic = BK_SIC.sel(member=member).sel(time=slice(start, end)).groupby("time.month")
        large_sic = NPL_SIC.sel(member=member).sel(time=slice(start, end)).groupby("time.month")

        denom = len(large_sic) / 4 #,len(small_sic))
        
        true_count_cwm = 0
        true_count_bk = 0
        true_count_np = 0

        
        for month in [12,1,2]:
            mask_t = temp[month] < climate_normal_pap.sel(month=month)
            mask_bk = small_sic[month] < baseline_bk * 0.9 
            mask_np = large_sic[month] < baseline_np * 0.9 

            combined_bk = mask_t & mask_bk 
            combined_np = mask_t & mask_np 

            true_count_cwm += mask_t.sum().values
            true_count_bk += combined_bk.sum().values
            true_count_np += combined_np.sum().values
            
        
        ratio_bk = true_count_bk / true_count_cwm
        ratio_np = true_count_np / true_count_cwm

        row.append(f"{( ratio_bk)*100:.1f}  ({( ratio_np)*100:.1f})")
        bk_ratio_for_mean[label].append(( ratio_bk)*100)
        np_ratio_for_mean[label].append(( ratio_np)*100)
    table.add_row(row)

mean_row = ["Mean"]
for label, _, _ in periods:
    if bk_ratio_for_mean[label]:
        mean_count = np.mean(bk_ratio_for_mean[label])
        mean_ratio = np.mean(np_ratio_for_mean[label])
        mean_row.append(f"{mean_count:.1f}  ({mean_ratio:.1f})")
    else:
        mean_row.append("nan(nan)")

table.add_row(mean_row)

print(table)
print(table.get_latex_string())


#### Time over CWM prob

In [ ]:
europe_temp_clim_pap = europe_temp.sel(time=slice("1971-01", "2000-12"))

combined_member_mean_pap = europe_temp_clim_pap.groupby("time.month").mean(dim="time")

climate_normal_pap = combined_member_mean_pap.mean(dim="member")  

periods = [
    ("1970-01", "1979-12"),
    ("1980-01", "1989-12"),
    ("1990-01", "1999-12"),
    ("2000-01", "2009-12"),
    ("2010-01", "2019-12"),
    ("2020-01", "2029-12"),
    ("2030-01", "2039-12"),
    ("2040-01", "2049-12"),
    ("2050-01", "2059-12"),
    ("2060-01", "2069-12"),
    ("2070-01", "2079-12"),
    ("2080-01", "2089-12"),
    ("2090-01", "2100-12"),
]

clim_start, clim_end = "1971-01", "2000-12"
clim_counts = {}
for member in members_list:
    temp_clim = europe_temp.sel(member=member).sel(time=slice(clim_start, clim_end))
    true_count = 0
    for month in [12,1,2]:
        mask = temp_clim.groupby("time.month")[month] < climate_normal_pap.sel(month=month)
        true_count += int(np.sum(mask.values))
    clim_counts[member] = true_count


prob = []  
for member in members_list:
    baseline = clim_counts[member]
    row_vals = []

    for start, end in periods:
        temp = europe_temp.sel(member=member).sel(time=slice(start, end))
        true_count = 0

        for month in [12, 1, 2]:
            mask = temp.groupby("time.month")[month] < climate_normal_pap.sel(month=month)
            true_count += int(mask.sum().values)

        if baseline != 0:
            row_vals.append(true_count / (len(mask) * 3) * 100)
        else:
            row_vals.append(np.nan)

    prob.append(row_vals)

values_1970s = [row[0] for row in prob]
values_1980s = [row[1] for row in prob]
values_1990s = [row[2] for row in prob]
values_2000s = [row[3] for row in prob]
values_2010s = [row[4] for row in prob]
values_2020s = [row[5] for row in prob]
values_2030s = [row[6] for row in prob]
values_2040s = [row[7] for row in prob]
values_2050s = [row[8] for row in prob]
values_2060s = [row[9] for row in prob]
values_2070s = [row[10] for row in prob]
values_2080s = [row[11] for row in prob]
values_2090s = [row[12] for row in prob]

CWM_decadal_mean = [np.mean(values_1970s),np.mean(values_1980s),np.mean(values_1990s),np.mean(values_2000s),np.mean(values_2010s),np.mean(values_2020s),
    np.mean(values_2030s),np.mean(values_2040s),np.mean(values_2050s),np.mean(values_2060s),np.mean(values_2070s),np.mean(values_2080s),np.mean(values_2090s)]

CWM_decadal_std = [np.std(values_1970s),np.std(values_1980s),np.std(values_1990s),np.std(values_2000s),np.std(values_2010s),np.std(values_2020s),
    np.std(values_2030s),np.std(values_2040s),np.std(values_2050s),np.std(values_2060s),np.std(values_2070s),np.std(values_2080s),np.std(values_2090s)]


In [ ]:
decade_labels = [
    "1970s","1980s","1990s","2000s","2010s",
    "2020s","2030s","2040s","2050s","2060s",
    "2070s","2080s","2090s"
]

plt.figure(figsize=(10, 5))
y_range = range(1,14)
plt.errorbar(
    y_range,
    CWM_decadal_mean,
    yerr=CWM_decadal_std,
    fmt='o',
    capsize=5,
    elinewidth=1,
    markeredgewidth=1
)

plt.xticks(y_range, decade_labels, rotation=45, ha='right')
plt.xlabel("Decade", fontsize=16)
plt.ylabel("CWM (%)", fontsize=16)
plt.title("CWM probability over decades", fontsize=20)

plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("CWMoverDecades.png", bbox_inches="tight", dpi=300)

#### BK SIC Winter 

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 8))

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

djf_t_ERA5 = temp_ERA5_ng.where(temp_ERA5_ng['valid_time.month'].isin([12,1,2]), drop=True)

djf = SIC_BK_SAT.where(SIC_BK_SAT['time.month'].isin([12, 1, 2]), drop=True)
djf_ec_mem = BK_SIC.where(BK_SIC['time.month'].isin([12, 1, 2]), drop=True)
djf_ec = djf_ec_mem.mean(dim='member')
djf_ec_t_mem = europe_temp.where(europe_temp['time.month'].isin([12, 1, 2]), drop=True)
djf_ec_t = djf_ec_t_mem.mean(dim='member')

yearly_sat_mean = djf.groupby("time.year").mean()
yearly_ec_mean  = djf_ec.groupby("time.year").mean()
yearly_ec_t_mean = djf_ec_t.groupby("time.year").mean()
yearly_ERA_mean = djf_t_ERA5.groupby("valid_time.year").mean()

ax[0].plot(djf_t_ERA5['valid_time'], djf_t_ERA5, '.', c='r', alpha=0.5,
           label='ERA5 reconstructed temperature data')
ax[0].plot(pd.to_datetime(yearly_ERA_mean['year'], format='%Y'),
           yearly_ERA_mean.values, 'o-', alpha=0.6, c='k',
           label='Yearly DJF temperature mean')
ax[0].plot(pd.to_datetime(yearly_ec_t_mean['year'], format='%Y'),
           yearly_ec_t_mean.values, 'o-', alpha=0.6, c='g',
           label='Yearly DJF EC-Earth3 mean')

ax[1].plot(djf.time, djf.values*100, '.', c='r', alpha=0.5,
           label='SIC Satellite values')
ax[1].plot(pd.to_datetime(yearly_sat_mean['year'], format='%Y'),
           yearly_sat_mean.values*100, 'o-', alpha=0.6, c='k',
           label='Yearly DJF mean')
ax[1].plot(pd.to_datetime(yearly_ec_mean['year'], format='%Y'),
           yearly_ec_mean.values, 'o-', alpha=0.6, c='g',
           label='Yearly DJF EC-Earth3 mean')

ax[1].set(xlim=[pd.to_datetime(1975, format='%Y'),
                pd.to_datetime(2028, format='%Y')],
          ylim=[15, 85])
ax[0].set(xlim=[pd.to_datetime(1975, format='%Y'),
                pd.to_datetime(2028, format='%Y')])

ax[0].set_ylabel("Temperature (°C)", fontsize=16)
ax[1].set_ylabel("BK SIC (%)", fontsize=16)
ax[0].set_xlabel("Year", fontsize=16)
ax[1].set_xlabel("Year", fontsize=16)

ax[0].set_title("DJF Mean Temperatures")
ax[1].set_title("DJF Barents–Kara SIC")

fig.suptitle("DJF Mean Temperature and Barents–Kara SIC Trends", fontsize=20)

for j in range(2):
    ax[j].legend()
    ax[j].grid()

plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("DJF_overview.png", bbox_inches="tight", dpi=50)

## Checking correlation



In [ ]:
def linear(x, a, b):
    return a * x + b

def chi2(a, b, x, y):
    model = linear(x, a, b)
    return np.sum((y - model)**2)  
    
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
          'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
members_list = (['r1', 'r4', 'r6', 'r9', 'r11', 'r13', 'r15', 'r101', 'r102', 'r103',
       'r104', 'r105', 'r106', 'r107', 'r108', 'r109', 'r110', 'r111', 'r112',
       'r113', 'r114', 'r115', 'r116', 'r117', 'r118', 'r119', 'r120', 'r121',
       'r122', 'r123', 'r124', 'r125', 'r126', 'r127', 'r128', 'r129', 'r130',
       'r131', 'r132', 'r133', 'r134', 'r135', 'r136', 'r137', 'r138', 'r139',
       'r140', 'r141', 'r142', 'r143', 'r144', 'r145', 'r146', 'r147', 'r148', 'r149', 'r150'])

cmap = cm.get_cmap('tab20', len(members_list))  

color_map = {member: mcolors.to_hex(cmap(i)) for i, member in enumerate(members_list)}

all_results = []

def BK_EU_plot(BK_data, eu_data, discard_0_SIC, experiment, lag_dir, experiment_name, plot_title, savefig):
    results = {
        "a": [],
        "b": [],
        "chi2_r": [],
        "r2": [],
        "rmse": [],        
        "pearson_r": [],   
        "p_value": [],     
        "n": [],
        "month": [],
        "member": [],
        "experiment": []
    }
    
    fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(16, 20)) 
    experiment_name = experiment_name
    std_cut = 1.5
    time_roll = 5

    for i in range(12):
        l = i // 3
        j = i % 3
        text_y = []
        text_x = []
        
        for member in members_list:

            temp_by_month = eu_data.sel(member=member).groupby("time.month")
            sic_by_month = BK_data.sel(member=member).groupby("time.month")

            if lag_dir == 'lag_eu':
                idx_y = i + 2 if i < 11 else 1  
                idx_x = i + 1 if i < 11 else 12 
                
                y = temp_by_month[idx_y]
                x = sic_by_month[idx_x]
                
                if idx_y < idx_x: 
                    x = x[:-1]  
                    y = y[1:]   
                    
            elif lag_dir == 'lag_bk':
                idx_y = i + 1 if i < 11 else 12
                idx_x = i + 2 if i < 11 else 1
                
                y = temp_by_month[idx_y]
                x = sic_by_month[idx_x]

                if idx_x < idx_y:
                    y = y[:-1]  
                    x = x[1:]   

            else:
                idx_y = i + 1
                idx_x = i + 1
                y = temp_by_month[idx_y]
                x = sic_by_month[idx_x]
            
            if experiment == 'roll':
                y_roll = y.rolling(time=time_roll, center=False).mean()
                y_roll_std = y.rolling(time=time_roll, center=False).std()
        
                x_roll = x.rolling(time=time_roll, center=False).mean()
                x_roll_std = x.rolling(time=time_roll, center=False).std()

                x = x_roll
                y = y_roll

                x = x.values
                y = y.values

            elif experiment == 'CWM':
                
                idx = y < climate_normal[idx_y - 1]
                y = y[idx.values].values
                x = x[idx.values].values
                
            elif experiment == 'warm' or experiment == 'cold':
                y_roll = y.rolling(time=time_roll, center=False).mean()
                y_roll_std = y.rolling(time=time_roll, center=False).std()
        
                x_roll = x.rolling(time=time_roll, center=False).mean()
                x_roll_std = x.rolling(time=time_roll, center=False).std()


                x = x.values
                y = y.values
                y_roll = y_roll.values

                y_roll_std = y_roll_std.values
                
                temp_norm_std = ((y - y_roll) / y_roll_std )
                if experiment == 'warm':
                    std_ind = temp_norm_std > std_cut
                else:
                    std_ind = temp_norm_std < -std_cut
                    
                x = x[std_ind]
                y = y[std_ind]
            else:
                x = x.values
                y = y.values
            


            if discard_0_SIC == False:
                valid_mask = ~np.isnan(x) & ~np.isnan(y)
            else:
                valid_mask = ~np.isnan(x) & ~np.isnan(y) & (x > 1)

            x =  np.array(x[valid_mask])
            y =  np.array(y[valid_mask])

            if len(y)  <= 2:
                continue
                
            m = Minuit(lambda a, b: chi2(a, b, x, y), a=0, b=4)
            m.migrad()

            a_fit, b_fit = m.values["a"], m.values["b"]
            chi2_fit = m.fval
            chi2_fit_r = chi2_fit / (len(y) - 2)

            y_fit = linear(np.array(x), a_fit, b_fit)

            ss_res = np.sum((y - y_fit)**2)
            ss_tot = np.sum((y - np.mean(y))**2)
            r2 = 1 - ss_res / ss_tot
            
            rmse = np.sqrt(ss_res / len(y))
            
            r_val, p_val = stats.pearsonr(x, y)
            
            ax[l, j].plot(x, y, '.', c=color_map[member]) 
            x_line = np.linspace(min(np.array(x)), np.max(np.array(x)), 5)
            ax[l, j].plot(x_line, linear(x_line, a_fit, b_fit), '-.', alpha=0.75, c=color_map[member], lw=1.5)
            
            text_y.append(np.min(y))
            text_x.append(np.max(x))

            results["a"].append(a_fit)
            results["b"].append(b_fit)
            results["chi2_r"].append(chi2_fit_r)
            results["r2"].append(r2)
            
            results["rmse"].append(rmse)
            results["pearson_r"].append(r_val)
            results["p_value"].append(p_val) # Optional
            
            results["n"].append(len(x))
            results["month"].append(months[i].lower())
            results["member"].append(member)
            results["experiment"].append(experiment_name)


        ax[l, j].set_title(f"{months[i].capitalize()}", fontsize=14)

        if len(text_x) > 0 and len(text_y) > 0:
            
            x_pos = np.max(text_x) * 0.6
            y_pos = np.min(text_y)
            
            current_month_indices = [k for k, x in enumerate(results["month"]) if x == months[i].lower()]
            
            if current_month_indices:
                current_slopes = [results["a"][k] for k in current_month_indices]
                current_intercepts = [results["b"][k] for k in current_month_indices]
                
                mean_slope = np.mean(current_slopes)
                std_slope = np.std(current_slopes)
                mean_int = np.mean(current_intercepts)
                std_int = np.std(current_intercepts)

                ax[l, j].text(x_pos, y_pos,
                    f"Slope   = {mean_slope:.3f} ± {std_slope:.3f}\n"
                    f"Intercept = {mean_int:.2f} ± {std_int:.2f}",
                    fontsize=12)
        else:
            ax[l, j].text(0.5, 0.5, "No Valid Data", transform=ax[l, j].transAxes, 
                          ha='center', va='center', color='red')

        ax[l, j].set_xlabel("BK SIC [%]", fontsize=12)
        ax[l, j].set_ylabel("Europe temp [°C]", fontsize=12)
        
        ax[l, j].grid()
        ax[l, j].xaxis.set_inverted(True)

    plt.suptitle(plot_title)
    plt.tight_layout()
    plt.subplots_adjust(top=0.94)
    plt.show()  
    if savefig == True:
        fig.savefig(f"{plot_title}.png", bbox_inches="tight", dpi=50)
    df = pd.DataFrame(results)
    ds = df.set_index(["experiment", "month", "member"]).to_xarray()
    all_results.append(df)

BK_EU_plot(BK_SIC_fut,europe_temp_fut,True,'CWM','lag_bk','CWM',' CWM BK vs EU temp',False)

In [ ]:
def linear(x, a, b):
    return a * x + b

def chi2(a, b, x, y):
    model = linear(x, a, b)
    return np.sum((y - model)**2)  

def BK_EU_plot_djf(BK_data, eu_data, discard_0_SIC, experiment, lag_dir, experiment_name, plot_title, savefig):
    results = {
        "a": [], "b": [], "chi2_r": [], "r2": [], "rmse": [],        
        "pearson_r": [], "p_value": [], "n": [], "month": [],
        "member": [], "experiment": []
    }
    
    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(18, 5)) 
    experiment_name = experiment_name
    std_cut = 1.5
    time_roll = 5

    djf_indices = [11, 0, 1]

    for plot_idx, i in enumerate(djf_indices):
        text_y = []
        text_x = []
        
        for member in members_list:
            temp_by_month = eu_data.sel(member=member).groupby("time.month")
            sic_by_month = BK_data.sel(member=member).groupby("time.month")

            if lag_dir == 'lag_eu':
                idx_y = i + 2 if i < 11 else 1  
                idx_x = i + 1 if i < 11 else 12 
                
                y = temp_by_month[idx_y]
                x = sic_by_month[idx_x]
                
                if idx_y < idx_x: 
                    x = x[:-1]  
                    y = y[1:]   

            elif lag_dir == 'lag_bk':
                idx_y = i + 1 if i < 11 else 12
                idx_x = i + 2 if i < 11 else 1
                
                y = temp_by_month[idx_y]
                x = sic_by_month[idx_x]

                if idx_x < idx_y:
                    y = y[:-1]  
                    x = x[1:]   

            else:
                idx_y = i + 1
                idx_x = i + 1
                y = temp_by_month[idx_y]
                x = sic_by_month[idx_x]
            
            if experiment == 'roll':
                y_roll = y.rolling(time=time_roll, center=False).mean()
                y_roll_std = y.rolling(time=time_roll, center=False).std()
        
                x_roll = x.rolling(time=time_roll, center=False).mean()
                x_roll_std = x.rolling(time=time_roll, center=False).std()

                x = x_roll
                y = y_roll

                x = x.values
                y = y.values

            elif experiment == 'CWM':
                idx = y < climate_normal[idx_y - 1]
                y = y[idx.values].values
                x = x[idx.values].values

            elif experiment == 'warm' or experiment == 'cold':
                y_roll = y.rolling(time=time_roll, center=False).mean()
                y_roll_std = y.rolling(time=time_roll, center=False).std()
        
                x_roll = x.rolling(time=time_roll, center=False).mean()
                x_roll_std = x.rolling(time=time_roll, center=False).std()

                x = x.values
                y = y.values
                y_roll = y_roll.values

                y_roll_std = y_roll_std.values
                
                temp_norm_std = ((y - y_roll) / y_roll_std )
                if experiment == 'warm':
                    std_ind = temp_norm_std > std_cut
                else:
                    std_ind = temp_norm_std < -std_cut
                    
                x = x[std_ind]
                y = y[std_ind]
            else:
                x = x.values
                y = y.values
            
            if discard_0_SIC == False:
                valid_mask = ~np.isnan(x) & ~np.isnan(y)
            else:
                valid_mask = ~np.isnan(x) & ~np.isnan(y) & (x > 1)

            x =  np.array(x[valid_mask])
            y =  np.array(y[valid_mask])

            if len(y)  <= 2:
                continue
                
            m = Minuit(lambda a, b: chi2(a, b, x, y), a=0, b=4)
            m.migrad()

            a_fit, b_fit = m.values["a"], m.values["b"]
            chi2_fit = m.fval
            chi2_fit_r = chi2_fit / (len(y) - 2)

            y_fit = linear(np.array(x), a_fit, b_fit)

            ss_res = np.sum((y - y_fit)**2)
            ss_tot = np.sum((y - np.mean(y))**2)
            r2 = 1 - ss_res / ss_tot
            
            rmse = np.sqrt(ss_res / len(y))
            
            r_val, p_val = stats.pearsonr(x, y)
            
            ax[plot_idx].plot(x, y, '.', c=color_map[member]) 
            x_line = np.linspace(min(np.array(x)), np.max(np.array(x)), 5)
            ax[plot_idx].plot(x_line, linear(x_line, a_fit, b_fit), '-.', alpha=0.75, c=color_map[member], lw=1.5)
            
            text_y.append(np.min(y))
            text_x.append(np.max(x))

            results["a"].append(a_fit)
            results["b"].append(b_fit)
            results["chi2_r"].append(chi2_fit_r)
            results["r2"].append(r2)
            results["rmse"].append(rmse)
            results["pearson_r"].append(r_val)
            results["p_value"].append(p_val) 
            
            results["n"].append(len(x))
            results["month"].append(months[i].lower())
            results["member"].append(member)
            results["experiment"].append(experiment_name)

        ax[plot_idx].set_title(f"{months[i].capitalize()}", fontsize=14)

        if len(text_x) > 0 and len(text_y) > 0:
            x_pos = np.max(text_x) * 0.6
            y_pos = np.min(text_y)
            
            current_month_indices = [k for k, val in enumerate(results["month"]) if val == months[i].lower()]
            
            if current_month_indices:
                current_slopes = [results["a"][k] for k in current_month_indices]
                current_intercepts = [results["b"][k] for k in current_month_indices]
                
                mean_slope = np.mean(current_slopes)
                std_slope = np.std(current_slopes)
                mean_int = np.mean(current_intercepts)
                std_int = np.std(current_intercepts)

                ax[plot_idx].text(x_pos, y_pos,
                    f"Slope   = {mean_slope:.3f} ± {std_slope:.3f}\n"
                    f"Intercept = {mean_int:.2f} ± {std_int:.2f}",
                    fontsize=12)
        else:
            ax[plot_idx].text(0.5, 0.5, "No Valid Data", transform=ax[plot_idx].transAxes, 
                          ha='center', va='center', color='red')

        ax[plot_idx].set_xlabel("BK SIC [%]", fontsize=12)
        ax[plot_idx].set_ylabel("Europe temp [°C]", fontsize=12)
        
        ax[plot_idx].grid()
        ax[plot_idx].xaxis.set_inverted(True)

    plt.suptitle(plot_title)
    plt.tight_layout()
    plt.subplots_adjust(top=0.88) 
    plt.show()  
    
    if savefig == True:
        fig.savefig(f"{plot_title}.png", bbox_inches="tight", dpi=50)
    df = pd.DataFrame(results)
    ds = df.set_index(["experiment", "month", "member"]).to_xarray()
    all_results.append(df)
test = 'BK vs EU'
BK_EU_plot(BK_SIC, europe_temp, True, 'No exp', 'No lag', 'Normal',test,True)


In [ ]:
# TESTING
# experiment_t =  ['warm','cold','roll','CWM','a'] 
# lag_dir = ['lag_eu','lag_bk','a']
# all_results = []

# for exp in experiment_t:
#     for lag in lag_dir:
#         print(f"{lag},{exp}")
#         a = f"{lag},{exp}"
#         BK_EU_plot(BK_SIC, europe_temp, True, exp, lag, f'{exp}' f'{lag_dir}',a)
        

### All months



In [ ]:
all_results = []

title = 'BK vs EU temp'
BK_EU_plot(BK_SIC, europe_temp, True, 'No exp', 'No lag', 'Normal',title,True)
# all_results

### Rolling mean all months

In [ ]:
title = 'BK vs EU temp 5 year mean'
BK_EU_plot(BK_SIC, europe_temp, True, 'roll', 'No lag', '5 year roll',title,True)

#### 1 std above (warmer) correlation
MAKE the above plots just with 1 std over/under 5 year running mean
    see if there is significant improvement in BKSIC & EUT relationship.
        need something independent of datapoints for eval (need p-values for this? or red chi2)
    

In [ ]:
title = 'BK vs EU temp warm extremes '
BK_EU_plot(BK_SIC, europe_temp, True, 'warm', 'No lag', 'Warmer extremes',title,True)

#### 1 std below (colder) correlation

In [ ]:
title = 'BK vs EU temp cold extremes '
BK_EU_plot(BK_SIC, europe_temp, True, 'cold', 'No lag', 'Colder extremes',title,True)

### Time offset

In [ ]:
title = 'BK (+1) vs EU temp  '
BK_EU_plot(BK_SIC, europe_temp, True, 'no', 'lag_bk', 'Normal (+1) BK',title,False)

In [ ]:
title = 'BK vs EU (+1) temp  '
BK_EU_plot(BK_SIC, europe_temp, True, 'no', 'lag_eu', 'Normal (+1) EU',title,False)

In [ ]:
title = 'BK (+1) vs EU temp 5-year mean'
BK_EU_plot(BK_SIC, europe_temp, True, 'roll', 'lag_bk', '5 year roll (+1) BK',title,False)
# BK_EU_plot(BK_SIC, europe_temp, True, 'roll', 'No lag', '5 year roll',title)

In [ ]:
title = 'BK vs EU (+1) temp 5-year mean '
BK_EU_plot(BK_SIC, europe_temp, True, 'roll', 'lag_eu', '5 year roll (+1) EU',title,False)
# BK_EU_plot(BK_SIC, europe_temp, True, 'roll', 'No lag', '5 year roll',title)

####  1 std above (warmer) correlation
##### Future SIC

In [ ]:
title = 'BK (+1) vs EU temp warm extremes '
BK_EU_plot(BK_SIC, europe_temp, True, 'warm', 'lag_bk', 'Warmer extremes (+1) BK',title,True)

##### Future EU temp

In [ ]:
title = 'BK vs EU (+1) temp warm extremes '
BK_EU_plot(BK_SIC, europe_temp, True, 'warm', 'lag_eu', 'Warmer extremes (+1) EU',title,True)

####  1 std below (colder) correlation
##### Future SIC

In [ ]:
title = 'BK (+1) vs EU temp cold extremes '
BK_EU_plot(BK_SIC, europe_temp, True, 'cold', 'lag_bk', 'Colder extremes (+1) BK',title,True)

##### Future EU temp

In [ ]:
title = 'BK vs EU (+1) temp cold extremes '
BK_EU_plot(BK_SIC, europe_temp, True, 'cold', 'lag_eu', 'Colder extremes (+1) EU',title,True)

#### CWMs

In [ ]:
title = 'CWM BK vs EU'
BK_EU_plot(BK_SIC, europe_temp, True, 'CWM', 'no lag', 'CWM',title,True)

In [ ]:
title = 'CWM (+1) BK vs  EU'
BK_EU_plot(BK_SIC, europe_temp, True, 'CWM', 'lag_bk', 'CWM (+1) BK',title,True)

In [ ]:
title = 'CWM BK vs (+1) EU'
BK_EU_plot(BK_SIC, europe_temp, True, 'CWM', 'lag_eu', 'CWM (+1) EU',title,True)

### Compare experiments 

In [ ]:
df_all = pd.concat(all_results, ignore_index=True)
df_all = df_all.drop_duplicates(subset=["experiment", "month", "member"])
ds_all = df_all.set_index(["experiment", "month", "member"]).to_xarray()
ds_all.month

In [ ]:
df_all[['a', 'b', 'month', 'experiment']].to_csv('experiments_lin_data.csv', index=False)

In [ ]:
ds_all[['a', 'b']].to_netcdf('experiments_lin_data.nc')

#### Compare experiments PearsonR

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['pearson_r'].sel(month = 'jan',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['pearson_r'].sel(month = 'feb',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['pearson_r'].sel(month = 'mar',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-1.05,0.25)

    ax[i].set(title=f"{months[i]}".capitalize(), ylabel='PearsonR')

fig.savefig("PearRJFM.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['pearson_r'].sel(month = 'apr',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['pearson_r'].sel(month = 'may',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['pearson_r'].sel(month = 'jun',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-1.05,0.25)

    ax[i].set(title=f"{months[i+3]}".capitalize(), ylabel='PearsonR')

fig.savefig("PearRAMJ.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['pearson_r'].sel(month = 'jul',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['pearson_r'].sel(month = 'aug',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['pearson_r'].sel(month = 'sep',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-1.05,0.25)

    ax[i].set(title=f"{months[i+6]}".capitalize(), ylabel='PearsonR')

fig.savefig("PearRJAS.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['pearson_r'].sel(month = 'oct',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['pearson_r'].sel(month = 'nov',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['pearson_r'].sel(month = 'dec',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-1.05,0.25)

    ax[i].set(title=f"{months[i+9]}".capitalize(), ylabel='PearsonR')

fig.savefig("PearROND.png", bbox_inches="tight", dpi=300)

#### Compare experiments rmse

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['rmse'].sel(month = 'jan',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['rmse'].sel(month = 'feb',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['rmse'].sel(month = 'mar',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-0.05,4.25)

    ax[i].set(title=f"{months[i]}".capitalize(), ylabel='RMSE')

fig.savefig("rmseJFM.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['rmse'].sel(month = 'apr',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['rmse'].sel(month = 'may',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['rmse'].sel(month = 'jun',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-0.05,4.25)

    ax[i].set(title=f"{months[i+3]}".capitalize(), ylabel='RMSE')

fig.savefig("rmseAMJ.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['rmse'].sel(month = 'jul',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['rmse'].sel(month = 'aug',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['rmse'].sel(month = 'sep',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-0.05,4.25)

    ax[i].set(title=f"{months[i+6]}".capitalize(), ylabel='RMSE')

fig.savefig("rmseJAS.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['rmse'].sel(month = 'oct',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['rmse'].sel(month = 'nov',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['rmse'].sel(month = 'dec',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set_ylim(-0.05,4.25)

    ax[i].set(title=f"{months[i+9]}".capitalize(), ylabel='RMSE')

fig.savefig("rmseOND.png", bbox_inches="tight", dpi=300)

#### Compare r2

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['r2'].sel(month = 'jan',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['r2'].sel(month = 'feb',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['r2'].sel(month = 'mar',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set(title=f"{months[i]}".capitalize(), ylabel='R² value')

fig.savefig("r2JFM.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['r2'].sel(month = 'apr',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['r2'].sel(month = 'may',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['r2'].sel(month = 'jun',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set(title=f"{months[i+3]}".capitalize(), ylabel='R² value')
fig.savefig("r2AMJ.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['r2'].sel(month = 'jul',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['r2'].sel(month = 'aug',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['r2'].sel(month = 'sep',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set(title=f"{months[i+6]}".capitalize(), ylabel='R² value')
fig.savefig("r2JAS.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['r2'].sel(month = 'oct',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['r2'].sel(month = 'nov',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['r2'].sel(month = 'dec',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    ax[i].set(title=f"{months[i+9]}".capitalize(), ylabel='R² value')
fig.savefig("r2OND.png", bbox_inches="tight", dpi=300)

#### Compare Pval

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['p_value'].sel(month = 'jan',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['p_value'].sel(month = 'feb',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['p_value'].sel(month = 'mar',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    # ax[i].set_yscale('log')
    ax[i].set(title=f"{months[i]}".capitalize(), ylabel='P value')

fig.savefig("p_valueJFM.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['p_value'].sel(month = 'apr',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['p_value'].sel(month = 'may',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['p_value'].sel(month = 'jun',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    # ax[i].set_yscale('log')
    ax[i].set(title=f"{months[i+3]}".capitalize(), ylabel='P value')

fig.savefig("p_valueAMJ.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['p_value'].sel(month = 'jul',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['p_value'].sel(month = 'aug',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['p_value'].sel(month = 'sep',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    # ax[i].set_yscale('log')
    ax[i].set(title=f"{months[i+6]}".capitalize(), ylabel='P value')

fig.savefig("p_valueJAS.png", bbox_inches="tight", dpi=300)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 6))
x_id = np.linspace(0,9,10)
for i in range(len(ds_all.experiment.values)):
    ax[0].boxplot(ds_all['p_value'].sel(month = 'oct',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[1].boxplot(ds_all['p_value'].sel(month = 'nov',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)
    ax[2].boxplot(ds_all['p_value'].sel(month = 'dec',experiment =ds_all.experiment.values[i] ).dropna(dim="member"),positions = [i],widths = 0.25)

for i in range(3):
    ax[i].set_xticks(range(len(ds_all.experiment.values)))
    ax[i].set_xticklabels(ds_all.experiment.values, rotation=65, ha='right')
    ax[i].grid()
    # ax[i].set_yscale('log')
    ax[i].set(title=f"{months[i+9]}".capitalize(), ylabel='P value')

fig.savefig("p_valueOND.png", bbox_inches="tight", dpi=300)